# 가지 자동 라벨링 — Kaggle GPU 워커 (v4.32.0)

Kaggle CPU 마스터가 `kaggle_gpu`로 배정된 요청(shape·track·자동 라벨·모델 비교·학습)을 보면 이 노트북을 **GPU를 켜고** 시작합니다.
할 일을 모두 마치고 5분 동안 새 요청이 없으면 스스로 끝나 GPU를 돌려줍니다. 사람이 Save & Run All을 누를 필요는 없습니다.

- 처음 한 번: 이 노트북을 Kaggle에 `eggplant-gpu-worker`(또는 routing.json `kaggle_kernels.gpu_worker_slug`)로 만들고, 보안 비밀 `RCLONE_CONF`·`CVAT_TOKEN`(쓰는 것 모두)을 붙이고, Accelerator를 GPU로 두세요.
- 설정(0번 셀의 `CVAT_PROFILES`·`OPERATOR`·`MODEL_CHOICE` 등)은 Colab 노트북과 같게 두세요. 이 노트북은 Drive `notebooks/`의 것을 CPU 마스터가 올립니다.
- 정지: CPU 마스터가 `jobs/STOP_GPU.json`을 남기면 하던 일만 마치고 끝납니다.


In [ ]:
# 0. 공통 설정 · 준비 — 세션이 새로 열릴 때마다 가장 먼저 실행 (다시 실행해도 안전) · 앱 버전 4.32.0 (Colab · Kaggle)
APP_VERSION = '4.32.0'             # 이 노트북의 버전 — Drive scripts/의 파일도 같은 버전이어야 함 (바꾸지 마세요)
# ====================== 설정 (여기만 고치세요) ======================
BASE     = '/content/drive/MyDrive/eggplant_yolo'   # 공유받은 폴더는 '내 드라이브'에 바로가기를 만들면 이 경로로 보임
CREATE_BASE = False            # True = BASE가 없을 때 새로 만듦 (처음 설치하는 사람만)
MODEL_CHOICE = 'latest'        # 쓸 학습 모델: 'latest' = runs/에서 가장 최근에 학습을 끝낸 모델 (15번 셀 결과가 자동으로 잡힘)
                               #   또는 아래 목록 번호(1, 2 …) · 또는 runs/ 폴더 이름 그대로 (예: 'bud_r1')
MODEL_NAMES = {                # 자주 쓰는 모델에 번호를 붙여 둠 (선택 사항)
    1: 'fruit_1920',                                   # 예전 dataset.zip으로 학습
    2: 'exp_20261025_081411_cvat_for_images_1_1',      # exports/20261025_081411_cvat_for_images_1_1로 학습 (15번 셀)
    3: 'fruit_r261101_2class_rotate',                  # 11/6 운영판: 밀린 과제 #3
}
MODEL    = 'yolo11s.pt'        # 사전학습 가중치 (s → 더 높이려면 m)
IMGSZ    = 1920                # 변환 때 TRAIN_IMGSZ와 같게
EPOCHS   = 100
PATIENCE = 30                  # 개선이 없으면 일찍 끝냄
BATCH    = -1                  # -1 = GPU 메모리에 맞춰 자동. 메모리 오류 시 2 또는 1
DEVICE   = None                # None = 자동(GPU 있으면 GPU)
TRACK_CONF = 0.25             # track 생성 시 탐지 신뢰도 하한. 검출이 적으면 0.1~0.15로 (추적기 기준도 함께 낮아짐)
TRACK_FILL_GAPS = 3            # 잠깐(태스크 프레임 3개 이하) 안 보인 열매는 track을 끊지 않고 CVAT 보간에 맡김
TRACK_BUFFER = 90              # 추적기가 가려진 열매의 ID를 기억하는 프레임 수 (30fps 기준 3초)
TRACK_DECODER = 'pyav'         # v4.31.4 기본 (밀린 과제 #4) · 영상 읽기: 'opencv' / 'pyav'(CVAT와 같은 방식) — 20번 셀 진단 결과에 따라
AUTO_TASK_REPLACE = True       # v4.31.4 (밀린 과제 #5): 이름이 _uld·_AA로 끝나는 자동 라벨 TASK는 기존 라벨을 백업한 뒤 교체가 기본
                               #   (5·6·10번 셀, 30번 셀) · 손 라벨 TASK는 각 셀의 ALLOW_REPLACE·TRACK_UPLOAD_REPLACE·AA_EXISTING 그대로 · False = 예전처럼
def auto_task(name):
    """자동 라벨 TASK(이름 끝 _uld · _AA)이고 AUTO_TASK_REPLACE가 켜져 있으면 True"""
    n = str(name or '').strip()
    return bool(globals().get('AUTO_TASK_REPLACE', False)) and (n.endswith('_uld') or bool(re.search(r'_AA(_r\d+)?$', n)))   # v4.31.5: AA_r1 = AA
TRACK_FRAME_OFFSET = 0         # 영상 프레임 + 이 값 = CVAT 프레임 (20번 셀 진단이 알려 줌)
TRACK_BOX_SOURCE = 'detection'  # 'detection' = 모델이 그 프레임에서 찾은 박스 (권장) / 'tracker' = 추적기 추정 박스(이전 방식, 늦게 따라와 잘림)
TRACK_NEAR = 1                 # 태스크 프레임에 검출이 없을 때 앞뒤 몇 프레임까지 빌려올지 (크면 카메라 이동만큼 밀림)
UPLOAD_MIN_PRECISION = 0.8     # 업로드 권장 기준 (검증 2단계 정밀도) — 대장에 '권장/재검토'로 표시
AA_CONF    = 0.25              # 프레임별 자동 라벨링 신뢰도 하한
REVIEWED_RULE = 'vf'          # 검토 완료 TASK = 이름의 상태에 이 글자가 있음 (vf, vf_r2, vf_r2df_ok, …_vf_AA). '' = 모두 검토 완료로 봄
                              #   검색어에서는 '검토'로 씀: '검토 date=261019..261022' (검색어 'vf'는 상태가 정확히 vf인 것만 — 다름)
VAL_RULE   = 'plant=3|plant=7'      # val(평가) TASK = 이 검색어에 맞는 TASK — 모든 온실·라인의 3번·7번 그루. 15번이 TASK 단위로 train/val을 나눔
                              #   '' = 예전처럼 무작위 영상(CONVERT의 VAL_RATIO) · 예: 'gh=G4', 'row=2 plant=6..10' (웹 학습 요청이 바꿀 수 있음)
CVAT_PROFILES = {             # 계정(프로필) 목록. 이름은 자유 — 셀 맨 위 LOGIN에 이 이름을 씀
    'token':    {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN'},                    # PAT (Google 로그인 계정)
    'password': {'host': 'https://app.cvat.ai', 'login': 'password', 'user': 'CVAT_USER', 'pass': 'CVAT_PASS'}, # 공동 아이디 (ID/PASS)
    'B':        {'host': 'https://app.cvat.ai', 'login': 'token',    'token': 'CVAT_TOKEN_B'},                  # B 계정 PAT (보안 비밀 CVAT_TOKEN_B)
}                              # 'token' / 'user' / 'pass' 값 = Colab 보안 비밀 이름 (계정마다 다른 이름으로)
CVAT_SDK_VERSION = 'auto'      # 'auto' = 처음 접속할 서버와 같은 X.Y 버전의 cvat-cli/cvat-sdk 설치 (SaaS 자동 업그레이드 대응)
                               # 문제가 생기면 확인된 버전으로 고정: 예 '2.79.0'
CVAT_AUTH  = 'auto'            # 처음 접속할 프로필: 'auto' = 보안 비밀이 있는 첫 프로필 · 또는 프로필 이름
                               # 셀마다 맨 위 LOGIN = '프로필 이름'으로 계정·서버를 바꿀 수 있음
CVAT_ORG     = 'horti'        # 기본 조직: CVAT 조직의 slug 또는 이름 (CVAT 화면 왼쪽 위 조직 메뉴에 보이는 것)
                               #   '' = 개인 작업공간 · None = 구분 안 함 (보이는 모든 조직의 TASK — v4.18까지와 같음)
CVAT_PROJECT = ''              # v4.31.4 기본 (밀린 과제 #2): eggplant·eggplant_AA 모두 — CPU 마스터는 config/app_settings.json (같게 둘 것) · 기본 프로젝트: 이름 또는 ID (그 조직 안). '' = 모든 프로젝트 · '-' = 프로젝트 없는 TASK만
                               #   정하면 셀의 TASK 목록·검색·백업·export가 이 범위만 보고, 16·17번 셀의 새 TASK도 이 조직·프로젝트에 만듦
                               #   TASK ID를 직접 적으면 프로젝트는 안 따짐(조직은 따짐). 프로필마다 다르게: CVAT_PROFILES 항목에 'org': …, 'project': …
                               #   웹 관리 화면의 요청은 그 TASK들의 조직으로 실행 (요청이 끝나면 다시 이 기본값)
DS_ROOT    = '/content/ds'     # 데이터셋을 풀 곳 (Colab 로컬 디스크 — 빠름)
OPERATOR   = ''               # 학습 잠금·완료 기록에 남길 내 이름 (예: 'A-재웅', 'B-james'). 사람마다 다르게
LOCK_STALE_MIN = 45           # 학습 잠금이 이 시간(분) 넘게 갱신되지 않으면 끊긴 학습으로 보고 넘겨받을 수 있음
LOCK_CONFIRM_SEC = 30         # 잠금을 잡은 뒤 이만큼 기다렸다 다시 확인 (두 컴퓨터가 거의 동시에 시작한 경우를 잡음)
# ---- Kaggle에서 실행할 때만 (v4.12) — Colab에서는 무시됨. 준비 방법은 KAGGLE.md ----
KAGGLE_REMOTE   = 'gdrive:'              # rclone 원격. root_folder_id = eggplant_yolo 폴더 ID로 만들었으면 'gdrive:' · 아니면 'gdrive:eggplant_yolo'
KAGGLE_BASE     = '/tmp/eggplant_yolo'   # Kaggle 안의 작업 폴더 (세션이 끝나면 사라짐 — 원본은 언제나 Drive)
KAGGLE_PULL     = ['exports']            # 0번 셀이 처음에 받아 둘 폴더. 영상·백업·TASK 폴더는 셀이 필요할 때 자동으로 받음
KAGGLE_PULL_DATASET = False              # True = dataset.zip · datasets/<선택 모델>.zip도 받음 (2·3번 셀용, 클 수 있음)
KAGGLE_PULL_ALL_BEST = False             # v4.31.6: False = 선택 모델(MODEL_CHOICE)의 best.pt만 받음, 다른 모델은 셀이 쓸 때 받음 · True = 예전처럼 모든 모델의 best.pt
KAGGLE_VIDEOS   = ''                     # 미리 받아 둘 영상 (TASK 이름 검색 규칙, 예 'G6 date=261019'). '' = 필요할 때만
KAGGLE_AUTO_SYNC = True                  # 셀마다: 실행 전 공유 기록 받기 → 실행 후 바뀐 파일을 Drive에 올리기 (끄면 drive_sync_now()로 직접)
# ====================================================================
import os, sys, glob, re, json, zipfile, shutil, subprocess, importlib, importlib.util
from pathlib import Path
from urllib.parse import urlparse
import time

# ---------- Kaggle에서 실행 (v4.12) — Drive를 붙일 수 없으므로 rclone으로 받고 올림. Colab에서는 아무것도 안 함 ----------
ON_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
KAGGLE_BATCH = os.environ.get('KAGGLE_KERNEL_RUN_TYPE') == 'Batch'      # Save & Run All(커밋)로 실행 중
LOCAL_TMP = '/tmp' if ON_KAGGLE else '/content'                         # 잠깐 쓰는 로컬 폴더
_KG = globals().get('_KG') or {}                                        # 이 세션의 동기화 상태 (0번 셀을 다시 실행해도 유지)
for _k, _v in {'ls': {}, 'media': set(), 'tasks': set(), 'lock_t': {}, 'bg': None, 'secrets': {}}.items():
    _KG.setdefault(_k, _v)
_KG_SHARED = ['logs/**', 'task_registry.csv', 'backup_index.csv', 'backup_status.csv',
              'backups/pending_backups.csv', 'exports/*.csv', 'requests/*.json']   # 셀마다 실행 전에 받는 공유 기록·웹 요청 (작음)
_KG_NOSYNC = ('raw_images/', 'cvat_frames/')                            # 프레임 캐시: 올리지도 받지도 않음 (다시 만들 수 있음)
_KG_NODELETE = ('videos/', 'backups/')                                  # Kaggle에서 지워도 Drive에서는 지우지 않음

def _kg_remote(rel=''):
    r = KAGGLE_REMOTE if KAGGLE_REMOTE.endswith(':') else KAGGLE_REMOTE.rstrip('/') + '/'
    return r + rel.strip('/')

_KG_RC_LOCK = globals().get('_KG_RC_LOCK') or __import__('threading').RLock()

def _kg_rc(args, quiet=False):
    # rclone 실행. 실패하면 마지막 오류만 보여 줌 (값·토큰은 출력에 나오지 않음)
    # v4.28.1: Drive에 쓰는 rclone은 한 번에 하나 — 작업자 셀의 심장 박동 스레드와 본 작업이 동시에 새 폴더를 만들면
    #          Google Drive에 같은 이름 폴더가 둘 생겨(relabel/ 등) 앱과 노트북이 서로 다른 쪽을 봄
    with _KG_RC_LOCK:
        r = subprocess.run(['rclone', *args], capture_output=True, text=True)
    if r.returncode and not quiet:
        err = '\n'.join(l for l in r.stderr.strip().splitlines() if 'ERROR' in l or 'Failed' in l or 'error' in l)[-600:]
        print(f'⚠️ rclone {args[0]} 실패 (코드 {r.returncode}): {err or r.stderr.strip()[-300:]}')
    return r

_KG_SHARDED = ('task_registry.csv', 'backup_index.csv', 'backups/pending_backups.csv', 'exports/pending_exports.csv',
               'logs/uploads.csv', 'logs/activity.csv', 'logs/train_tasks.csv', 'logs/model_compare.csv')

def _kg_filters(paths):
    # 'runs/a' → 파일 또는 폴더 전체 · '*'가 있으면 rclone 규칙 그대로 ('*'는 / 를 넘지 않음, '**'는 넘음)
    out = []
    for p in paths:
        p = '/' + p.strip('/')
        out += ['--include', p] if '*' in p else ['--include', p, '--include', p + '/**']
    if any(str(p).strip('/') in _KG_SHARDED for p in paths):     # v4.31.0: 그 기록의 세션 파일도 함께 (logs/shards/)
        out += ['--include', '/logs/shards/**']
    return out

def drive_pull(*paths, quiet=False):
    # Drive → Kaggle (BASE 기준 경로·패턴). 로컬이 더 새것이면 건너뜀. Colab에서는 아무것도 안 함
    if not ON_KAGGLE or not paths:
        return
    flt = _kg_filters(paths) + [x for n in _KG_NOSYNC for x in ('--exclude', f'/{n}**')]
    r = _kg_rc(['copy', _kg_remote(), BASE, '--update', '--transfers', '8', '--checkers', '16', '-v', '--stats', '0', *flt], quiet=quiet)
    for _l in (r.stderr or '').splitlines():       # 셀 안에서 Drive에서 받은 파일 → 셀이 끝날 때 다시 올리지 않음 (v4.24.2)
        _m = re.search(r'INFO\s+:\s+(.+?): Copied', _l)
        if _m:
            try:                                   # 받은 그대로인지 알려고 크기·수정 시각을 기억 (셀이 그 뒤에 고치면 올림)
                _st = os.stat(os.path.join(BASE, _m.group(1).strip()))
                _KG.setdefault('pulled', {})[_m.group(1).strip()] = (_st.st_mtime_ns, _st.st_size)
            except OSError:
                pass

def kg_sync_exports():
    # Kaggle: Drive exports/에서 새로 생긴 zip을 받아 둠 — 0번 셀 뒤에 다른 세션·컴퓨터가 받은 export도 보이게 (v4.23)
    if ON_KAGGLE:
        drive_pull('exports', quiet=True)

def drive_push(*paths, quiet=False):
    # Kaggle → Drive (BASE 기준 경로·패턴). Drive가 더 새것이면 건너뜀. 지우지는 않음
    if not ON_KAGGLE or not paths:
        return
    _kg_rc(['copy', BASE, _kg_remote(), '--update', '--transfers', '8', '--checkers', '16', *_kg_filters(paths)], quiet=quiet)

def drive_push_bg(*paths):
    # 학습 중 에폭마다: 앞의 올리기가 끝났을 때만 백그라운드로 올림 (학습을 멈추지 않음)
    import threading
    t = _KG.get('bg')
    if not ON_KAGGLE or (t and t.is_alive()):
        return
    _KG['bg'] = threading.Thread(target=drive_push, args=paths, kwargs={'quiet': True}, daemon=True)
    _KG['bg'].start()

def _kg_files(prefix):
    # Drive의 폴더 목록 (경로, 크기, 수정 시각) — 세션마다 한 번만 읽음
    if prefix not in _KG['ls']:
        r = _kg_rc(['lsf', '-R', '--files-only', '--format', 'pst', '--separator', '\t', _kg_remote(prefix)], quiet=True)
        _KG['ls'][prefix] = [tuple(l.split('\t')) for l in r.stdout.splitlines() if l.count('\t') == 2]
    return _KG['ls'][prefix]

_KG_VEXT = ('.mp4', '.mov', '.avi', '.mkv')

def drive_get_videos(query='', files=()):
    # Drive videos/에서 영상을 받음: query = TASK 이름 검색 규칙(파일 이름에 적용) · files = videos/ 기준 경로
    if not ON_KAGGLE:
        return []
    allv = [(p, s) for p, s, t in _kg_files('videos') if p.lower().endswith(_KG_VEXT)]
    pick = [p for p, s in allv if p in set(files)]
    if query:
        pick += [p for p, s in allv if ctt.match_name(os.path.splitext(os.path.basename(p))[0], query)[0]]
    pick = sorted(set(pick))
    if pick:
        mb = sum(int(s) for p, s in allv if p in pick) / 2**20
        print(f'⬇️ 영상 {len(pick)}개 받기 ({mb:.0f} MB) …')
        drive_pull(*[f'videos/{p}' for p in pick])
    return pick

def kg_file_index(*prefixes):
    # Kaggle: Drive 파일 목록 {BASE 기준 경로: (크기, 수정 시각 epoch)} — 새로 읽음 (모델 정리·웹 상태, v4.24.8). Colab은 None
    if not ON_KAGGLE:
        return None
    import datetime as _dt
    out = {}
    for pre in prefixes:
        _KG['ls'].pop(pre, None)
        for p, s, t in _kg_files(pre):
            try:
                ts = _dt.datetime.strptime(t[:19], '%Y-%m-%d %H:%M:%S').timestamp()
            except ValueError:
                ts = 0
            out[f'{pre}/{p}'] = (int(s), ts)
    return out

def kaggle_media_on_drive(source='', tname=''):
    # TASK 원본 영상이 Drive에 있는지만 봄 (받지 않음, v4.24.7): 'videos/<경로>' · 'backups/<경로>' · ''
    if not ON_KAGGLE:
        return ''
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [p for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        return f'videos/{vids[0]}'
    bk = sorted((t, p) for p, s, t in _kg_files('backups') if base and p.startswith(base + '/') and p.endswith('.zip'))
    return f'backups/{bk[-1][1]}' if bk else ''

def kaggle_fetch_media(source='', tname=''):
    # TASK 하나의 원본 영상을 Drive에서 받음: videos/(같은 이름) → 없으면 backups/<기본 이름>/의 가장 최근 백업
    key = (source or '', tname or '')
    if not ON_KAGGLE or key in _KG['media'] or _KG.get('no_fetch'):
        return
    _KG['media'].add(key)
    base = ctt.base_name(tname) if (ctt is not None and tname) else ''
    want = {x.lower() for x in (base, os.path.splitext(os.path.basename(source or ''))[0], tname or '') if x}
    vids = [(p, s) for p, s, t in _kg_files('videos')
            if p.lower().endswith(_KG_VEXT) and os.path.splitext(os.path.basename(p))[0].lower() in want]
    if vids:
        todo = [p for p, s in vids if not os.path.exists(f'{BASE}/videos/{p}')]
        if todo:
            print(f"⬇️ 영상 받기: {', '.join(todo)} ({sum(int(s) for p, s in vids if p in todo) / 2**20:.0f} MB)")
            drive_pull(*[f'videos/{p}' for p in todo])
        return
    if base:
        bk = sorted((t, p, s) for p, s, t in _kg_files('backups') if p.startswith(base + '/') and p.endswith('.zip'))
        if bk and not os.path.exists(f'{BASE}/backups/{bk[-1][1]}'):
            print(f'⬇️ 백업 받기 (영상 포함): backups/{bk[-1][1]} ({int(bk[-1][2]) / 2**20:.0f} MB)')
            drive_pull(f'backups/{bk[-1][1]}')

def _kg_lock_mirror(run_dir, force=False):
    # Drive의 RUNNING.json을 그대로 비춤 (없으면 로컬도 지움) — 학습 잠금이 다른 컴퓨터와 통하도록
    rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
    if not force and time.time() - _KG['lock_t'].get(rel, 0) < 15:
        return rel
    _KG['lock_t'][rel] = time.time()
    r = _kg_rc(['cat', _kg_remote(rel)], quiet=True)
    p = f'{BASE}/{rel}'
    if r.returncode == 0 and r.stdout.strip():
        os.makedirs(os.path.dirname(p), exist_ok=True)
        open(p, 'w', encoding='utf-8').write(r.stdout)
    elif os.path.exists(p):
        os.remove(p)
    return rel

def _kg_lock_publish(rel):
    p = f'{BASE}/{rel}'
    if os.path.exists(p):
        _kg_rc(['copyto', p, _kg_remote(rel)], quiet=True)
    else:
        _kg_rc(['deletefile', _kg_remote(rel)], quiet=True)
    _KG['lock_t'][rel] = time.time()

def _kg_patch_ctt():
    # ctt(공용 도구)의 잠금·영상 찾기 함수에 Drive 동기화를 덧붙임 (0번 셀이 ctt를 다시 불러올 때마다)
    if not ON_KAGGLE or ctt is None:
        return
    if hasattr(ctt, 'find_video'):
        _fv = ctt.find_video
        def find_video(*a, **k):
            try:
                kaggle_fetch_media(k.get('source', a[1] if len(a) > 1 else ''),
                                   k.get('task_name', k.get('tname', a[2] if len(a) > 2 else '')))
            except Exception as e:
                print(f'(Drive에서 영상 받기 실패: {e})')
            return _fv(*a, **k)
        ctt.find_video = find_video
    if hasattr(ctt, 'video_from_backups'):
        _vb = ctt.video_from_backups
        def video_from_backups(*a, **k):
            try:
                kaggle_fetch_media('', k.get('task_name', a[1] if len(a) > 1 else ''))
            except Exception as e:
                print(f'(Drive에서 백업 받기 실패: {e})')
            return _vb(*a, **k)
        ctt.video_from_backups = video_from_backups
    if all(hasattr(ctt, f) for f in ('acquire_run_lock', 'touch_run_lock', 'release_run_lock', 'read_run_lock')):
        _acq, _touch, _rel, _read = ctt.acquire_run_lock, ctt.touch_run_lock, ctt.release_run_lock, ctt.read_run_lock
        def read_run_lock(run_dir, *a, **k):
            _kg_lock_mirror(run_dir)
            return _read(run_dir, *a, **k)
        def acquire_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            drive_pull(os.path.relpath(run_dir, BASE), quiet=True)     # 이어서 학습할 last.pt·results.csv도 받음
            res = _acq(run_dir, *a, **k)
            if res and res[0]:
                _kg_lock_publish(rel)
            return res
        def touch_run_lock(run_dir, *a, **k):
            rel = os.path.relpath(os.path.join(run_dir, 'RUNNING.json'), BASE)
            due = time.time() - _KG['lock_t'].get(rel, 0) >= 60          # Drive와 주고받기는 1분에 한 번
            if due:
                _kg_lock_mirror(run_dir, force=True)
            res = _touch(run_dir, *a, **k)
            if due and res:
                _kg_lock_publish(rel)
            return res
        def release_run_lock(run_dir, *a, **k):
            rel = _kg_lock_mirror(run_dir, force=True)
            res = _rel(run_dir, *a, **k)
            _kg_lock_publish(rel)
            return res
        ctt.read_run_lock, ctt.acquire_run_lock = read_run_lock, acquire_run_lock
        ctt.touch_run_lock, ctt.release_run_lock = touch_run_lock, release_run_lock

def _kg_snapshot():
    snap = {}
    for root, dirs, files in os.walk(BASE):
        rel_root = os.path.relpath(root, BASE)
        if rel_root != '.' and (rel_root + '/').startswith(_KG_NOSYNC):
            dirs[:] = []
            continue
        for f in files:
            p = os.path.join(root, f)
            try:
                st = os.stat(p)
            except FileNotFoundError:
                continue
            snap[os.path.relpath(p, BASE)] = (st.st_mtime_ns, st.st_size)
    return snap

def _kg_pre_cell(*_a):
    if not (ON_KAGGLE and KAGGLE_AUTO_SYNC):
        return
    drive_pull(*_KG_SHARED, quiet=True)
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()

def drive_sync_now(echo=True):
    # 지난 스냅숏 뒤에 바뀐 파일을 Drive에 올리고, 셀에서 지운 파일은 Drive에서도 지움(휴지통). 셀이 끝날 때 자동 실행
    snap0 = _KG.pop('snap', None)
    if not ON_KAGGLE or snap0 is None:
        return
    import tempfile
    snap1 = _kg_snapshot()
    _KG.pop('snap_t', 0)
    pulled = _KG.pop('pulled', {})               # 셀 안에서 Drive에서 받은 뒤 그대로인 파일은 빼고
    mine = lambda p: not p.endswith(('.partial', 'RUNNING.json'))   # 학습 잠금은 잠금 함수가 따로 주고받음
    # 셀이 새로 만들거나 바꾼 파일 전부 — 수정 시각을 보존한 복사(shutil.copy2: 14번 셀 export 재사용)도 올림 (v4.24.2, 예전엔 빠졌음)
    changed = sorted(p for p, v in snap1.items() if snap0.get(p) != v and pulled.get(p) != v and mine(p))
    deleted = sorted(p for p in snap0 if p not in snap1 and not p.startswith(_KG_NODELETE) and mine(p))
    for lst, args in ((changed, ['copy', BASE, _kg_remote(), '--update', *(['--no-traverse'] if len(changed) <= 200 else []), '--transfers', '8']),
                      (deleted, ['delete', _kg_remote()])):
        if lst:
            with tempfile.NamedTemporaryFile('w', suffix='.txt', delete=False, encoding='utf-8') as fh:
                fh.write('\n'.join(lst) + '\n')
            _kg_rc(args + ['--files-from-raw', fh.name])
            os.remove(fh.name)
    if echo and (changed or deleted):
        mb = sum(snap1[p][1] for p in changed) / 2**20
        print('☁️ Drive 반영: ' + ' · '.join(x for x in (f'올림 {len(changed)}개 ({mb:.1f} MB)' if changed else '',
                                                    f'지움 {len(deleted)}개 (Drive 휴지통)' if deleted else '') if x))

def _kg_post_cell(*_a):
    if ON_KAGGLE and KAGGLE_AUTO_SYNC:
        try:
            drive_sync_now()
        except Exception as e:
            print(f'⚠️ Drive 올리기 실패: {e} → drive_sync_now() 또는 drive_push(경로)로 다시')

def _kg_secret(name):
    # 읽은 값만 기억 — 없던 비밀을 나중에 붙이면 0번 셀을 다시 실행하는 것만으로 읽힘 (v4.13)
    if not _KG['secrets'].get(name):
        try:
            from kaggle_secrets import UserSecretsClient
            _KG['secrets'][name] = UserSecretsClient().get_secret(name)
        except Exception:
            return None
    return _KG['secrets'][name]

def _kg_fix_conf(conf):
    # Kaggle 보안 비밀의 rclone.conf를 되살림 (v4.31.4-1: CPU 마스터와 같은 처리기)
    # 한 줄로 붙은 것 · 글자 그대로 남은 백슬래시-n · 토큰 JSON 뒤에 붙은 team_drive 등 · 원격 이름이 gdrive가 아닌 것
    NL, CR, BS = chr(10), chr(13), chr(92)
    s = conf.replace(CR + NL, NL).replace(CR, NL).replace(BS + 'n', NL).strip()
    if s[:1] in ('"', "'") and s[-1:] == s[:1]:
        s = s[1:-1].strip()
    blobs = []
    def _mask_all(t):                       # 괄호 짝을 세어 JSON({...}, 안의 [...] 포함)을 통째로 가림 (v4.31.4-2)
        out_, i, n = [], 0, len(t)
        while i < n:
            if t[i] != '{':
                out_.append(t[i]); i += 1; continue
            d, j, q = 0, i, None
            while j < n:
                c = t[j]
                if q:
                    if c == BS:
                        j += 1
                    elif c == q:
                        q = None
                elif c == '"':
                    q = c
                elif c == '{':
                    d += 1
                elif c == '}':
                    d -= 1
                    if d == 0:
                        break
                j += 1
            blobs.append(t[i:j + 1]); out_.append(f'__JSON_BLOB_{len(blobs) - 1}__'); i = j + 1
        return ''.join(out_)
    s = _mask_all(s)
    out, sec = [], None
    for part in re.split(r'(?:^|(?<=\s))(?=\[[A-Za-z0-9_.-]+\])', s):
        part = part.strip()
        m = re.match(r'^\[([A-Za-z0-9_.-]+)\]\s*(.*)$', part, flags=re.S)
        if not m:
            continue
        sec, body = m.group(1), m.group(2)
        kv = []
        for ch in re.split(r'(?:^|\s+)(?=[A-Za-z0-9_]+\s*=)', body.strip()):
            ch = ch.strip()
            if '=' not in ch:
                continue
            k, v = (x.strip() for x in ch.split('=', 1))
            for i, bl in enumerate(blobs):
                v = v.replace(f'__JSON_BLOB_{i}__', bl)
            if k == 'token' and v:
                obj, _ = json.JSONDecoder().raw_decode(v[v.index('{'):])   # 뒤에 남은 '}' 등은 버림
                v = json.dumps(obj, separators=(',', ':'))
            kv.append(f'{k} = {v}')
        out += [f'[{sec}]'] + kv + ['']
        if sec != 'gdrive' and len([x for x in out if x.startswith('[')]) == 1:
            out += ['[gdrive]'] + kv + ['']       # KAGGLE_REMOTE 'gdrive:'로도 찾게
    if not out:
        raise SystemExit("⛔ RCLONE_CONF에서 [원격 이름] 줄을 찾지 못했습니다 → 보안 비밀 값이 '[gdrive]'로 시작하는지 확인 (KAGGLE.md 2장)")
    return NL.join(out).strip() + NL

def _kg_install_rclone():
    # v4.31.5: ① 노트북에 붙인 Kaggle 데이터셋의 rclone 파일(/kaggle/input/**/rclone)을 복사 (1~2초)
    #          ② 없으면 내려받기 (브라우저 이름을 붙이고 60초 제한) ③ 그것도 안 되면 GitHub 최신판 ④ apt
    if shutil.which('rclone'):
        return
    import urllib.request, io, glob as _g
    _t0 = time.time()
    dst = '/usr/local/bin/rclone' if os.access('/usr/local/bin', os.W_OK) else os.path.expanduser('~/.local/bin/rclone')
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    def _done(how):
        os.chmod(dst, 0o755)
        os.environ['PATH'] = os.path.dirname(dst) + ':' + os.environ['PATH']
        print(f'rclone 설치: {how} ({time.time() - _t0:.0f}초)')
    src = next((p for p in _g.glob('/kaggle/input/**/rclone', recursive=True) if os.path.isfile(p)), None)
    if src:
        shutil.copy(src, dst)
        return _done(f'붙인 데이터셋 {os.path.dirname(src)}')
    def _get(url, timeout):
        req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 Chrome/126.0 Safari/537.36'})
        data = urllib.request.urlopen(req, timeout=timeout).read()
        with zipfile.ZipFile(io.BytesIO(data)) as z:
            open(dst, 'wb').write(z.read([n for n in z.namelist() if n.endswith('/rclone')][0]))
    try:
        _get('https://downloads.rclone.org/rclone-current-linux-amd64.zip', 60)
        return _done('downloads.rclone.org')
    except Exception as e:
        print(f'(rclone 받기 실패 downloads.rclone.org: {e})')
    try:                                          # GitHub는 첫 주소가 안 될 때만 물어봄
        tag = json.load(urllib.request.urlopen('https://api.github.com/repos/rclone/rclone/releases/latest', timeout=20))['tag_name']
        _get(f'https://github.com/rclone/rclone/releases/download/{tag}/rclone-{tag}-linux-amd64.zip', 120)
        return _done('github.com')
    except Exception as e:
        print(f'(rclone 받기 실패 github.com: {e})')
    if shutil.which('apt-get'):
        subprocess.run(['apt-get', 'install', '-y', '-qq', 'rclone'], capture_output=True)
        if shutil.which('rclone'):
            print(f'rclone 설치: apt ({time.time() - _t0:.0f}초)')
            return
    raise SystemExit('⛔ rclone을 설치하지 못했습니다 → Kaggle Session options에서 Internet이 On인지 확인하거나, '
                     'rclone 파일을 Kaggle 데이터셋으로 올려 이 노트북에 붙이세요 (Add Input)')

if ON_KAGGLE:
    BASE = KAGGLE_BASE
    if DS_ROOT.startswith('/content'):
        DS_ROOT = f'{LOCAL_TMP}/ds'
    os.makedirs(BASE, exist_ok=True)
    _kg_install_rclone()
    _conf = _kg_secret('RCLONE_CONF')              # v4.31.10 (R2): Kaggle 보안 비밀만 (환경 변수는 읽지 않음)
    if not _conf:
        raise SystemExit('⛔ Kaggle 보안 비밀 RCLONE_CONF가 없습니다 → Add-ons → Secrets에 rclone.conf 내용을 넣고, 이 노트북에 체크해 붙이세요 (KAGGLE.md 2장)')
    _conf = _kg_fix_conf(_conf)
    # v4.31.10 (R2): 홈(~/.config)이 아닌 세션 임시 폴더에 처음부터 본인만 읽게(0600) 만들고 RCLONE_CONFIG로 연결
    if not _KG.get('sec_dir') or not os.path.isdir(_KG['sec_dir']):
        import tempfile
        _KG['sec_dir'] = tempfile.mkdtemp(prefix='eggplant_sec_')
    _cp = Path(_KG['sec_dir']) / 'rclone.conf'
    if not _cp.exists() or _cp.read_text() != _conf:
        _fd = os.open(str(_cp), os.O_WRONLY | os.O_CREAT | os.O_TRUNC, 0o600)
        with os.fdopen(_fd, 'w') as _f:
            _f.write(_conf)
    os.chmod(_cp, 0o600)
    os.environ['RCLONE_CONFIG'] = str(_cp)
    _t0 = time.time()
    _kg_t = []                                     # v4.31.6: 단계별 걸린 시간 (0번 셀이 느릴 때 어디서인지)
    _r = _kg_rc(['lsf', '--max-depth', '1', _kg_remote('scripts')], quiet=True)
    _kg_t.append(f'Drive 확인 {time.time() - _t0:.0f}초')
    if _r.returncode != 0:
        _names = re.findall(r'^\[(.+?)\]', _conf, re.M)
        raise SystemExit(f'⛔ Drive {_kg_remote("scripts")} 를 읽지 못했습니다.\n'
                         f'   rclone 오류: {_r.stderr.strip()[-700:]}\n'
                         f'   설정의 원격 이름 {_names} · KAGGLE_REMOTE {KAGGLE_REMOTE!r}\n'
                         '   → notFound: root_folder_id 확인 · accessNotConfigured: GCP에서 Drive API 사용 · '
                         'invalid_grant: 토큰 만료(OAuth 앱을 프로덕션으로, PC에서 rclone config reconnect gdrive:) (KAGGLE.md 2장)')
    for _what, _ps in (('scripts·config', ('scripts', 'config')),
                       ('학습 기록', ('runs/*/*', 'datasets/*.sig', 'web/label_stats.json')),   # 가중치는 받지 않음 (선택 모델 best.pt는 아래에서)
                       ('모든 best.pt', ('runs/*/weights/best.pt',) if KAGGLE_PULL_ALL_BEST else ()),
                       ('공유 기록', tuple(_KG_SHARED)), ('KAGGLE_PULL', tuple(KAGGLE_PULL)),
                       ('dataset.zip', ('dataset.zip',) if KAGGLE_PULL_DATASET else ())):
        if _ps:
            _ts = time.time()
            drive_pull(*_ps)
            _kg_t.append(f'{_what} {time.time() - _ts:.0f}초')
    if KAGGLE_VIDEOS:
        _ts = time.time()
        drive_get_videos(KAGGLE_VIDEOS)
        _kg_t.append(f'영상 {time.time() - _ts:.0f}초')
    print(f'Kaggle: Drive에서 받기 {time.time() - _t0:.0f}초 ({" · ".join(_kg_t)}) → {BASE}')
    _KG['snap'], _KG['snap_t'] = _kg_snapshot(), time.time()   # 0번 셀이 바꾸는 파일(logs/versions.csv 등)도 끝에 올림
    try:
        _ip = get_ipython()
        for _ev, _fn in (('pre_run_cell', _kg_pre_cell), ('post_run_cell', _kg_post_cell)):
            for _old in [f for f in _ip.events.callbacks[_ev] if getattr(f, '__name__', '') == _fn.__name__]:
                _ip.events.unregister(_ev, _old)
            _ip.events.register(_ev, _fn)
    except NameError:
        pass


# Drive 연결
if not ON_KAGGLE and not os.path.exists('/content/drive/MyDrive') and BASE.startswith('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')
def _find_base(base):
    # BASE가 없으면 공유 바로가기·공유 드라이브에서 같은 이름의 폴더를 찾음
    name = os.path.basename(base.rstrip('/'))
    cands = glob.glob(f'/content/drive/MyDrive/**/{name}', recursive=False) + \
            glob.glob(f'/content/drive/.shortcut-targets-by-id/*/{name}') + \
            glob.glob(f'/content/drive/Shareddrives/*/{name}')
    return [c for c in cands if os.path.isdir(f'{c}/scripts')]
if not os.path.isdir(BASE):
    found = _find_base(BASE)
    if found:
        print(f'BASE {BASE} 가 없어 찾은 폴더를 씁니다: {found[0]}')
        BASE = found[0]
    elif CREATE_BASE:
        print(f'새 BASE 폴더를 만듭니다: {BASE}')
    else:
        raise SystemExit(
            f'⛔ {BASE} 폴더가 없습니다.\n'
            '   공유받은 폴더라면: Google Drive → 공유 문서함 → eggplant_yolo 오른쪽 클릭 → 정리 → 바로가기 추가 → 내 드라이브\n'
            '   그다음 메뉴 런타임 → 세션 다시 시작 → 0번 셀 다시. (처음 설치하는 사람이면 CREATE_BASE = True)')
if os.path.isdir(BASE) and not glob.glob(f'{BASE}/scripts/*.py'):
    found = [c for c in _find_base(BASE) if glob.glob(f'{c}/scripts/*.py') and os.path.realpath(c) != os.path.realpath(BASE)]
    msg = (f'⚠️ {BASE}/scripts 에 .py가 없습니다. 공유받은 폴더가 아니라 새로 생긴 빈 폴더일 수 있습니다.\n'
           '   빈 eggplant_yolo 폴더를 지우거나 이름을 바꾼 뒤, 공유받은 폴더의 바로가기를 내 드라이브에 추가하세요.')
    if found:
        msg += f'\n   코드가 있는 같은 이름의 폴더: {found[0]} → BASE를 이 경로로 바꿔도 됩니다.'
    print(msg)
for d in ['scripts', 'config', 'videos', 'runs', 'tasks', 'backups', 'logs']:
    os.makedirs(f'{BASE}/{d}', exist_ok=True)

# 패키지: 없을 때만 설치 (세션마다 확인)
from importlib import metadata
def _ver(pkg):
    try:
        return metadata.version(pkg)
    except metadata.PackageNotFoundError:
        return None

CVAT_HOST = CVAT_PROFILES[CVAT_AUTH if CVAT_AUTH in CVAT_PROFILES else next(iter(CVAT_PROFILES))]['host']   # 처음 접속할 서버 (셀에서 바뀜)

def cvat_server_version(host=None):
    # CVAT 서버 버전 (/api/server/about, 로그인 불필요). 실패하면 None
    try:
        import requests
        r = requests.get(f'{host or CVAT_HOST}/api/server/about', timeout=15)
        return r.json().get('version') if r.ok else None
    except Exception:
        return None

CVAT_LAST_KNOWN = '2.79.0'                   # 서버 버전을 못 읽을 때 쓸, 마지막으로 시험한 버전
SERVER_VER = cvat_server_version()
if CVAT_SDK_VERSION == 'auto':
    _base = SERVER_VER or CVAT_LAST_KNOWN
    _mj, _mn = _base.split('.')[:2]
    cvat_spec, cvat_prefix = f'cvat-cli~={_mj}.{_mn}.0', f'{_mj}.{_mn}.'   # 같은 X.Y의 최신 패치
else:
    cvat_spec, cvat_prefix = f'cvat-cli=={CVAT_SDK_VERSION}', CVAT_SDK_VERSION
_cli, _sdk = _ver('cvat-cli'), _ver('cvat-sdk')
need = [p for p, mod in [('ultralytics', 'ultralytics'), ('lap', 'lap'), ('av', 'av')] if importlib.util.find_spec(mod) is None]
if not (_cli and _sdk and _cli == _sdk and _cli.startswith(cvat_prefix)):
    need.append(cvat_spec)
if need and not os.environ.get('SKIP_INSTALL'):
    print('설치:', need)
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *need])
    if r.returncode != 0 and cvat_spec in need:
        # SaaS가 PyPI보다 먼저 올라간 경우 등: 같은 X.Y가 아직 없으면 최신 cvat-cli
        print(f'⚠️ {cvat_spec} 설치 실패 → 최신 cvat-cli 설치')
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', 'cvat-cli',
                        *[p for p in need if p != cvat_spec]], check=True)
    importlib.invalidate_caches()
    if 'cvat_sdk' in sys.modules and sys.modules['cvat_sdk'].__version__ != _ver('cvat-sdk'):
        print('⚠️ 이 세션에 이전 cvat-sdk가 이미 불러져 있습니다 → 메뉴 런타임 → 세션 다시 시작 후 0번 셀을 다시 실행하세요')
elif need:
    print('설치 필요(SKIP_INSTALL이라 건너뜀):', need)
import torch, ultralytics
CVAT_CLI_VER, CVAT_SDK_VER = _ver('cvat-cli'), _ver('cvat-sdk')

def cvat_compat(server_ver=None):
    # cvat-sdk는 서버가 SDK와 같은 X.Y 또는 한 단계 위(X.Y+1)일 때 공식 지원
    server_ver = server_ver or SERVER_VER
    if not (server_ver and CVAT_SDK_VER):
        return '확인 불가'
    s, k = [int(x) for x in server_ver.split('.')[:2]], [int(x) for x in CVAT_SDK_VER.split('.')[:2]]
    if s == k:
        return '일치'
    if s[0] == k[0] and s[1] == k[1] + 1:
        return '지원 범위 (서버가 한 단계 위)'
    return '⚠️ 지원 범위 밖 — CVAT_SDK_VERSION을 서버 버전에 맞추거나 auto로'

# 버전이 바뀔 때만 logs/versions.csv에 한 줄 (문제가 생겼을 때 언제부터인지 추적)
_vrow = [SERVER_VER or '?', CVAT_SDK_VER or '?', CVAT_CLI_VER or '?', ultralytics.__version__]
_vlog = f'{BASE}/logs/versions.csv'
os.makedirs(f'{BASE}/logs', exist_ok=True)
_last = open(_vlog, encoding='utf-8').read().strip().splitlines()[-1].split(',')[1:] if os.path.exists(_vlog) else None
if _last != _vrow:
    import datetime
    with open(_vlog, 'a', encoding='utf-8') as fh:
        if _last is None:
            fh.write('time,cvat_server,cvat_sdk,cvat_cli,ultralytics\n')
        from zoneinfo import ZoneInfo
        fh.write(','.join([datetime.datetime.now(ZoneInfo('Asia/Seoul')).strftime('%Y-%m-%d %H:%M')] + _vrow) + '\n')

# 데이터셋: dataset.zip이 바뀌었을 때만 다시 풂
DS, yaml_path = None, None
zpath = f'{BASE}/dataset.zip'
stamp = f'{DS_ROOT}/.source_mtime'
if os.path.exists(zpath):
    mt = str(os.path.getmtime(zpath))
    if not (os.path.exists(stamp) and open(stamp).read() == mt):
        shutil.rmtree(DS_ROOT, ignore_errors=True)
        zipfile.ZipFile(zpath).extractall(DS_ROOT)
        open(stamp, 'w').write(mt)
        print('데이터셋 풀기 완료')
    yaml_path = glob.glob(f'{DS_ROOT}/**/data.yaml', recursive=True)[0]
    DS = os.path.dirname(yaml_path)
    txt = open(yaml_path, encoding='utf-8').read()
    txt = re.sub(r'^path:.*$', f'path: {DS}', txt, flags=re.M)     # PC 경로 → Colab 경로
    open(yaml_path, 'w', encoding='utf-8').write(txt)

# 쓸 모델 정하기 (MODEL_CHOICE)
_trained = sorted(glob.glob(f'{BASE}/runs/*/TRAINING_DONE'), key=os.path.getmtime)
if MODEL_CHOICE == 'latest':
    NAME = os.path.basename(os.path.dirname(_trained[-1])) if _trained else MODEL_NAMES.get(1, 'fruit_1920')
elif isinstance(MODEL_CHOICE, int):
    assert MODEL_CHOICE in MODEL_NAMES, f'MODEL_CHOICE {MODEL_CHOICE}가 MODEL_NAMES 번호 {list(MODEL_NAMES)}에 없습니다'
    NAME = MODEL_NAMES[MODEL_CHOICE]
else:
    NAME = str(MODEL_CHOICE)

# 그 모델을 15번 셀로 학습했다면, 그때 만든 데이터셋(datasets/<NAME>.zip)을 DS로 → 2·3번 셀이 같은 데이터로 확인
if ON_KAGGLE and KAGGLE_PULL_DATASET:
    drive_pull(f'datasets/{NAME}.zip')           # Kaggle: 선택 모델의 데이터셋
_ds_zip = f'{BASE}/datasets/{NAME}.zip'
DS_SOURCE = 'dataset.zip' if DS else '없음'
if os.path.exists(_ds_zip):
    _ds_dir = f'{DS_ROOT}_exp/{NAME}'
    if not os.path.exists(f'{_ds_dir}/data.yaml'):
        shutil.rmtree(_ds_dir, ignore_errors=True)
        zipfile.ZipFile(_ds_zip).extractall(_ds_dir)
    yaml_path, DS = f'{_ds_dir}/data.yaml', _ds_dir
    _t = open(yaml_path, encoding='utf-8').read()
    open(yaml_path, 'w', encoding='utf-8').write(re.sub(r'^path:.*$', f'path: {DS}', _t, flags=re.M))
    DS_SOURCE = f'datasets/{NAME}.zip'

# 경로와 상태
RUN_DIR = f'{BASE}/runs/{NAME}'
BEST, LAST, DONE = f'{RUN_DIR}/weights/best.pt', f'{RUN_DIR}/weights/last.pt', f'{RUN_DIR}/TRAINING_DONE'
if ON_KAGGLE and not os.path.exists(BEST):        # v4.31.6: 선택 모델의 best.pt만 받음 (다른 모델은 셀이 쓸 때 need_file로)
    _ts = time.time()
    drive_pull(f'runs/{NAME}/weights/best.pt', quiet=True)
    print(f'Kaggle: 선택 모델 {NAME}/best.pt 받기 {time.time() - _ts:.0f}초' + ('' if os.path.exists(BEST) else ' — Drive에 없음'))
SCRIPTS = f'{BASE}/scripts'
VIDEOS, TASKS_DIR, REGISTRY = f'{BASE}/videos', f'{BASE}/tasks', f'{BASE}/task_registry.csv'
if SCRIPTS not in sys.path:
    sys.path.insert(0, SCRIPTS)
try:
    import cvat_task_tools as ctt
    importlib.reload(ctt)
except ImportError:
    ctt = None
if ctt is not None and hasattr(ctt, 'REVIEW_WORDS'):
    ctt.REVIEWED_RULE = REVIEWED_RULE                # 검색어 '검토' = 이 규칙의 검토 완료 TASK (v4.24)
try:                                            # 웹 관리 화면 연동 (v4.14): 상태 파일·웹 요청
    import web_bridge as wb
    importlib.reload(wb)
except ImportError:
    wb = None
_kg_patch_ctt()                                 # Kaggle: 영상 찾기·학습 잠금이 Drive와 통하게 (Colab에서는 아무것도 안 함)
if ON_KAGGLE and KAGGLE_VIDEOS and ctt is not None:
    drive_get_videos(KAGGLE_VIDEOS)

def epochs_done():
    p = f'{RUN_DIR}/results.csv'
    return max(0, sum(1 for _ in open(p)) - 1) if os.path.exists(p) else 0

def train_status():
    if os.path.exists(DONE):
        return '완료'
    if os.path.exists(LAST):
        return f'중단됨 ({epochs_done()}/{EPOCHS} 에폭) → 1번 셀이 이어서 학습'
    return '시작 전'

def _secret(name):
    # Colab 보안 비밀(왼쪽 열쇠 아이콘) → 없으면 None. 값은 출력하지 않음
    # Kaggle에서는 Kaggle 보안 비밀(Add-ons → Secrets), 둘 다 없으면 환경 변수
    # v4.31.9 (F11): Colab은 Colab 보안 비밀만 · Kaggle은 Kaggle 보안 비밀만 (환경 변수는 둘 다 아닌 로컬 실행에서만)
    v = None
    if ON_KAGGLE:
        v = _kg_secret(name)
    else:
        try:
            from google.colab import userdata
            try:
                v = userdata.get(name)
            except Exception:
                v = None
        except ImportError:
            v = os.environ.get(name)
    return v.strip().strip("'\"") if v else None

def _may_prompt():
    # v4.31.9 (F11): 보안 비밀이 없을 때 입력창으로 받는 것은 Colab에서 사람이 셀을 직접 실행할 때만 (Kaggle·작업자 실행 중에는 멈춤)
    w = globals().get('_W')
    return not ON_KAGGLE and not (isinstance(w, dict) and not w.get('stop', True))

def _need_secret(what, m):
    if not _may_prompt():
        raise SystemExit(f"⛔ [{m}] {what}이(가) 보안 비밀에 없습니다 → {'Kaggle Add-ons → Secrets' if ON_KAGGLE else 'Colab 보안 비밀'}에 넣고 이 노트북에 붙이세요 "
                         "(Kaggle·작업자 실행에서는 직접 입력을 받지 않음)")

def prof_login(name):
    # 프로필의 로그인 방식 ('token' 또는 'password')
    p = CVAT_PROFILES[name]
    return p.get('login') or ('token' if 'token' in p else 'password')

def _has_secret(name):
    p = CVAT_PROFILES[name]
    return bool(_secret(p['token'])) if prof_login(name) == 'token' else bool(_secret(p['user']) and _secret(p['pass']))

def cvat_mode(mode=None):
    # 쓸 프로필: 셀의 LOGIN → 지금 접속 중인 프로필 → 0번 셀 CVAT_AUTH ('auto' = 보안 비밀이 있는 첫 프로필)
    m = mode or globals().get('CVAT_LOGIN_MODE') or CVAT_AUTH
    if m == 'auto':
        m = next((k for k in CVAT_PROFILES if _has_secret(k)), next(iter(CVAT_PROFILES)))
    assert m in CVAT_PROFILES, f"LOGIN은 'auto' 또는 0번 셀 CVAT_PROFILES의 이름 {list(CVAT_PROFILES)} 중 하나 (지금 값: {m!r})"
    return m

_CRED_TYPED = globals().get('_CRED_TYPED', {})   # 보안 비밀이 없어 입력창으로 받은 값 (이 세션 메모리에만, 0번 셀을 다시 실행해도 유지)

def cvat_auth(mode=None):
    # 프로필의 (로그인 방식, 로그인 정보): ('password', (아이디, 비밀번호)) 또는 ('token', PAT)
    from getpass import getpass
    m = cvat_mode(mode)
    prof, kind = CVAT_PROFILES[m], prof_login(m)
    if kind == 'password':
        u = _secret(prof['user']) or _CRED_TYPED.get((m, 'u')) or (_need_secret(prof['user'], m) or input(f"[{m}] CVAT 아이디 ({prof['host']}): "))
        p = _secret(prof['pass']) or _CRED_TYPED.get((m, 'p')) or (_need_secret(prof['pass'], m) or getpass(f'[{m}] CVAT 비밀번호: '))
        _CRED_TYPED[(m, 'u')], _CRED_TYPED[(m, 'p')] = u, p
        return kind, (u, p)
    t = _secret(prof['token']) or _CRED_TYPED.get((m, 't')) or (_need_secret(prof['token'], m) or getpass(f"[{m}] CVAT Personal Access Token ({prof['host']}): "))
    _CRED_TYPED[(m, 't')] = t
    return kind, re.sub(r'^(Token|Bearer)\s+', '', t.strip(), flags=re.IGNORECASE)

def cvat_client(mode=None):
    # cvat-sdk 클라이언트 (그 프로필의 서버·계정으로 로그인)
    m = cvat_mode(mode)
    kind, v = cvat_auth(m)
    host = CVAT_PROFILES[m]['host']
    cl = ctt.connect(host, credentials=v) if kind == 'password' else ctt.connect(host, v)
    apply_scope(cl, m)                          # 기본 조직·프로젝트 (v4.19)
    return cl

_SCOPE_TMP = {}                                  # 웹 요청이 정한 이번 작업만의 조직 (요청이 끝나면 비움)
CVAT_SCOPE = globals().get('CVAT_SCOPE') or {}   # 지금 쓰는 조직·프로젝트 (use_cvat가 정함)

def scope_want(mode=None):
    # 쓸 조직·프로젝트: 웹 요청 → 프로필의 'org'·'project' → 0번 셀 CVAT_ORG·CVAT_PROJECT
    p = CVAT_PROFILES.get(cvat_mode(mode), {})
    if 'org' in _SCOPE_TMP:
        return _SCOPE_TMP['org'], _SCOPE_TMP.get('project', '')
    return p.get('org', globals().get('CVAT_ORG')), p.get('project', globals().get('CVAT_PROJECT', ''))

def apply_scope(cl, mode=None):
    # 클라이언트에 조직·프로젝트 범위를 붙임 (못 찾으면 있는 조직·프로젝트 목록과 함께 멈춤)
    org, proj = scope_want(mode)
    if not hasattr(ctt, 'set_scope'):
        return None
    try:
        return ctt.set_scope(cl, org, proj)
    except ValueError as e:
        raise SystemExit(f'⛔ {e}\n→ 0번 셀 CVAT_ORG·CVAT_PROJECT (또는 프로필의 org·project)를 고치고 0번 셀을 다시 실행하세요')

def cvat_cli_auth(mode=None):
    # cvat-cli에 넘길 인증 인자와 환경 변수 (비밀번호·토큰은 환경 변수로만 전달)
    kind, v = cvat_auth(mode)
    return (['--auth', v[0]], {'PASS': v[1]}) if kind == 'password' else ([], {'CVAT_ACCESS_TOKEN': v})

_CVAT_SESSIONS = globals().get('_CVAT_SESSIONS', {})   # 프로필별 접속 (0번 셀을 다시 실행해도 유지, 로그인 정보가 바뀌면 다시 로그인)

def cvat_forget(name=None):
    # 저장된 접속을 지움 (name=None이면 전부) → 다음 셀에서 다시 로그인
    for k in ([name] if name else list(_CVAT_SESSIONS)):
        _CVAT_SESSIONS.pop(k, None)
    for k in [k for k in _CRED_TYPED if name is None or k[0] == name]:
        _CRED_TYPED.pop(k, None)

def use_cvat(login=None):
    # 이 셀이 쓸 계정·서버를 정함: LOGIN=None이면 지금 그대로, 프로필 이름이면 그 계정으로 바꿈
    global CLIENT, CVAT_LOGIN_MODE, CVAT_HOST, CVAT_SERVER, CVAT_SCOPE
    import hashlib
    m = cvat_mode(login)
    host, kind = CVAT_PROFILES[m]['host'], prof_login(m)
    cred_sig = hashlib.sha256(repr((host,) + cvat_auth(m)).encode()).hexdigest()[:16]   # 값 자체는 저장 안 함
    sess = _CVAT_SESSIONS.get(m)
    if sess is None or sess[3] != cred_sig:
        if sess is not None:
            print(f'[{m}] 로그인 정보가 바뀌어 다시 로그인합니다.')
        def _login():
            cl_ = cvat_client(m)
            return cl_, cl_.api_client.users_api.retrieve_self()[0].username     # 로그인이 실제로 되는지 확인
        try:                                     # v4.31.4: 503·429·연결 오류는 1·2·4·8분 뒤 다시 (토큰 문제 아님)
            cl, user = ctt.with_retry(_login, what=f'CVAT 접속 [{m}]') if hasattr(ctt, 'with_retry') else _login()
        except Exception as e:
            e = e.__cause__ or e
            body = str(getattr(e, 'body', '') or '')[:200]
            _sh = ctt.cvat_error_hint(e) if hasattr(ctt, 'cvat_error_hint') else ''
            hint = _sh if _sh and '인증' not in _sh else (f"PAT가 만료됐거나 Read only인지, 보안 비밀 {CVAT_PROFILES[m].get('token')} 이름·값이 맞는지 확인하세요."
                    if kind == 'token' else
                    f"아이디·비밀번호가 틀렸습니다(보안 비밀 {CVAT_PROFILES[m].get('user')}·{CVAT_PROFILES[m].get('pass')} 값, 앞뒤 공백, "
                    f"비밀번호 변경 여부). 입력창으로 넣었다면 cvat_forget('{m}') 후 다시. "
                    'Google 로그인으로 만든 계정은 비밀번호가 없으므로 PAT 프로필로 접속하세요.')
            status = getattr(e, 'status', '')
            raise RuntimeError(f'CVAT 접속 실패 [{m} → {host}] ({type(e).__name__} {status}: {body or str(e)[:200]})\n→ {hint}') from None
        sv = cvat_server_version(host)
        _CVAT_SESSIONS[m] = (cl, f'{user}@{urlparse(host).hostname}', sv, cred_sig)
        if sv and cvat_compat(sv).startswith('⚠️'):
            print(f'⚠️ 이 서버({sv})는 설치된 cvat-sdk {CVAT_SDK_VER}의 지원 범위 밖입니다. 일부 기능이 안 될 수 있습니다.')
    CLIENT, CVAT_SERVER, _sv, _ = _CVAT_SESSIONS[m]
    CVAT_LOGIN_MODE, CVAT_HOST = m, host
    CVAT_SCOPE = apply_scope(CLIENT, m) or {}
    print(f"▶ CVAT 서버: {CVAT_SERVER} · 프로필 {m} ({'PAT' if kind == 'token' else 'ID/PASS'}) · 버전 {_sv or '?'}")
    if hasattr(ctt, 'scope_text'):
        print(f"  범위: {ctt.scope_text(CVAT_SCOPE)}" + (' ← 웹 요청의 조직 (이 작업만)' if 'org' in _SCOPE_TMP else ''))
    return CLIENT

def task_dir(tid):
    # TASK별 작업 폴더. app.cvat.ai가 아닌 서버는 앞에 서버 이름을 붙여 TASK ID가 겹쳐도 섞이지 않게
    host = urlparse(CVAT_HOST).hostname or ''
    tag = '' if host == 'app.cvat.ai' else re.sub(r'[^A-Za-z0-9]+', '-', host) + '_'
    d = f'{TASKS_DIR}/{tag}{tid}'
    if ON_KAGGLE and d not in _KG['tasks']:      # Kaggle: 이 TASK의 작업 폴더를 처음 쓸 때 Drive에서 받음
        _KG['tasks'].add(d)
        drive_pull(os.path.relpath(d, BASE), quiet=True)
    return d

def need_file(p):
    # v4.31.6 Kaggle: BASE 안의 파일(다른 모델의 best.pt 등)이 없으면 그때 Drive에서 받음 → p · Colab은 그대로
    if ON_KAGGLE and p and str(p).startswith(BASE + '/') and not os.path.exists(str(p)):
        drive_pull(os.path.relpath(str(p), BASE), quiet=True)
    return p

def task_reviewed(name):
    # 검토 완료 TASK인가 (0번 셀 REVIEWED_RULE: 이름의 상태에 그 글자가 있으면)
    return ctt.is_reviewed(name, REVIEWED_RULE) if ctt is not None else True

def task_role(name):
    # 'val' / 'train' / '규칙 밖'(이름 규칙에 안 맞음 → 학습에서는 train) / ''(VAL_RULE 없음)
    return ctt.split_role(name, VAL_RULE) if ctt is not None else ''

def load_script(name):
    # Drive scripts/의 .py를 노트북 안으로 불러옴 (자동 실행 안 됨 → 설정을 바꾼 뒤 함수 호출)
    spec = importlib.util.spec_from_file_location(os.path.splitext(name)[0], f'{SCRIPTS}/{name}')
    mod = importlib.util.module_from_spec(spec)
    sys.modules[spec.name] = mod
    spec.loader.exec_module(mod)
    return mod

def run(cmd, log=None, env=None):
    # 명령 실행: 화면에 보여 주고, log 경로가 있으면 같은 내용을 파일에도 남김. env = 추가 환경 변수(화면에 안 나옴)
    print('$', ' '.join(map(str, cmd)))
    p = subprocess.Popen(list(map(str, cmd)), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                         env={**os.environ, 'PYTHONUNBUFFERED': '1', **(env or {})})   # 진행 출력을 바로바로 보여 줌
    fh = open(log, 'a', encoding='utf-8') if log else None
    if fh:
        fh.write('$ ' + ' '.join(map(str, cmd)) + '\n')
    for line in p.stdout:
        print(line, end='')
        if fh:
            fh.write(line)
    p.wait()
    if fh:
        fh.close()
    return p.returncode

def model_label():
    # 대장에 남길 모델 표시: runs/<NAME> + best.pt 수정 시각
    if not os.path.exists(BEST):
        return ''
    import datetime
    return f"{NAME}/best.pt ({datetime.datetime.fromtimestamp(os.path.getmtime(BEST)).strftime('%m-%d %H:%M')})"

# ---------- 학습 잠금 (v4.9): 같은 runs/<이름>에 두 사람이 동시에 쓰지 않게 ----------
import uuid, time
if 'SESSION_ID' not in globals():
    SESSION_ID = uuid.uuid4().hex[:8]          # 이 Colab 세션의 표시 (0번 셀을 다시 실행해도 그대로)
if ctt is not None and hasattr(ctt, 'set_shard'):   # v4.31.0: 함께 쓰는 기록은 세션마다 따로 (logs/shards/<이름>/<세션>.csv) → 여러 세션이 동시에 써도 줄이 사라지지 않음
    ctt.set_shard(SESSION_ID)
TAKEOVER_LOCK = globals().get('TAKEOVER_LOCK', False)

def who_am_i():
    return OPERATOR or f'이름없음-{SESSION_ID}'

def lock_run(run_dir, stage):
    # 잠금을 잡음. 다른 사람이 학습 중이면 멈춤(SystemExit)
    held = (ctt.read_run_lock(run_dir) or {}).get('session') == SESSION_ID   # 이미 이 세션이 잡고 있음
    ok, msg = ctt.acquire_run_lock(run_dir, SESSION_ID, who_am_i(), stage, LOCK_STALE_MIN, TAKEOVER_LOCK)
    if not ok:
        raise SystemExit(msg)
    if msg:
        print('ℹ️', msg)
    if LOCK_CONFIRM_SEC and not held:
        print(f'학습 잠금: {os.path.basename(run_dir)} ({who_am_i()}, 세션 {SESSION_ID}) — {LOCK_CONFIRM_SEC}초 뒤 다시 확인 …')
        time.sleep(LOCK_CONFIRM_SEC)
        cur = ctt.read_run_lock(run_dir)
        if cur and cur.get('session') != SESSION_ID:
            raise SystemExit(f"⛔ 거의 같은 때 {cur.get('who')}(세션 {cur.get('session')})도 이 이름으로 시작했습니다 → 이 세션은 멈춥니다. "
                             '다른 TRAIN_NAME을 쓰세요.')

def unlock_run(run_dir):
    ctt.release_run_lock(run_dir, SESSION_ID)

def _lock_callbacks(model, run_dir):
    # 에폭마다 잠금 갱신. 다른 세션이 잠금을 가져갔으면 즉시 멈춤 (결과가 섞이지 않게)
    def hb(trainer):
        ep = int(getattr(trainer, 'epoch', -1)) + 1
        if not ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=ep):
            raise RuntimeError(f'⛔ 다른 세션이 runs/{os.path.basename(run_dir)} 잠금을 가져갔습니다 → 이 학습을 멈춥니다 (결과 섞임 방지)')
    for ev in ('on_pretrain_routine_end', 'on_train_epoch_start', 'on_fit_epoch_end'):
        model.add_callback(ev, hb)
    if ON_KAGGLE:                                # Kaggle: 에폭마다 runs/<이름>을 Drive에 올림 (세션이 끊겨도 이어서 학습)
        model.add_callback('on_fit_epoch_end', lambda tr: drive_push_bg(os.path.relpath(run_dir, BASE)))

def _train_multi_gpu(m, kw, run_dir, n_done, start):
    # v4.30: GPU 여럿으로 동시 학습 (Ultralytics DDP). 학습은 다른 프로세스에서 돌아 에폭 콜백(잠금 갱신·Drive 올리기)이 닿지 않음
    #   → 여기서 1분마다 잠금을 갱신하고 (Kaggle) runs/<이름>을 Drive로. 시작하지 못하면(에폭 0) GPU 하나로 다시 학습
    import threading as _th
    stop = _th.Event()
    def beat():
        while not stop.wait(60):
            try:
                ctt.touch_run_lock(run_dir, SESSION_ID, who=who_am_i(), stage='train', epoch=n_done())
                if ON_KAGGLE:
                    drive_push_bg(os.path.relpath(run_dir, BASE))
            except Exception:
                pass
    _th.Thread(target=beat, daemon=True).start()
    print(f"GPU {kw['device'].count(',') + 1}개로 동시 학습 (device={kw['device']}, batch={kw.get('batch')}) — 에폭마다 결과는 runs/{os.path.basename(run_dir)}/results.csv")
    try:
        m.train(**kw)
    except Exception as e:
        if n_done() > 0:
            raise
        print(f"⚠️ GPU 여럿으로 학습을 시작하지 못했습니다 ({type(e).__name__}: {str(e)[:200]}) → GPU 하나로 다시 (batch = 0번 셀 BATCH)")
        kw.update(device=0, batch=BATCH)
        from ultralytics import YOLO
        m2 = YOLO(start or MODEL); _lock_callbacks(m2, run_dir); m2.train(**kw)
    finally:
        stop.set()

def run_training(name, data_yaml, data_sig='', start=None, train_args=None):
    # start = 시작 가중치 (None = 0번 셀 MODEL) · train_args = Ultralytics 학습 설정 덮어쓰기 (lr0·optimizer·epochs 등, v4.24.3)
    # 잠금을 잡고 학습 (완료=건너뜀, 중단=이어서, 처음=새로). 1번·15번 셀이 함께 씀
    from ultralytics import YOLO
    run_dir = f'{BASE}/runs/{name}'
    last, done = f'{run_dir}/weights/last.pt', f'{run_dir}/TRAINING_DONE'
    if os.path.exists(done):
        print('이미 학습 완료 → 건너뜁니다.', open(done, encoding='utf-8').read())
        return False
    lock_run(run_dir, 'train')
    try:
        sp = f'{run_dir}/DATASET_SIG'
        if data_sig:
            old = open(sp).read().strip() if os.path.exists(sp) else ''
            if old and old != data_sig:
                raise SystemExit(f'⛔ runs/{name}은(는) 다른 데이터({old})로 학습하던 이름입니다 (지금 {data_sig}) → TRAIN_NAME을 바꾸세요')
            open(sp, 'w').write(data_sig)
        n_done = lambda: max(0, sum(1 for _ in open(f'{run_dir}/results.csv')) - 1) if os.path.exists(f'{run_dir}/results.csv') else 0
        try:
            if os.path.exists(last):
                print(f'이어서 학습: {n_done()}/{EPOCHS} 에폭부터')
                m = YOLO(last); _lock_callbacks(m, run_dir); m.train(resume=True)
            else:
                m = YOLO(start or MODEL); _lock_callbacks(m, run_dir)
                _kw = dict(data=data_yaml, imgsz=IMGSZ, epochs=EPOCHS, batch=BATCH, patience=PATIENCE,
                           device=DEVICE, project=f'{BASE}/runs', name=name, exist_ok=True)
                _kw.update(train_args or {})
                if isinstance(_kw.get('device'), str) and ',' in _kw['device']:
                    _train_multi_gpu(m, _kw, run_dir, n_done, start)       # v4.30: GPU 여럿 (Kaggle T4×2)
                else:
                    m.train(**_kw)
        except AssertionError as e:            # 이미 끝난 학습을 resume하려 할 때
            if 'nothing to resume' not in str(e) and 'is finished' not in str(e):
                raise
            print('학습이 이미 끝나 있었습니다.')
        import datetime as _d
        open(done, 'w', encoding='utf-8').write(f'{n_done()} epochs · data {name} · {who_am_i()} · 세션 {SESSION_ID} · '
                                                f'{_d.datetime.now().strftime("%Y-%m-%d %H:%M")}')
        return True
    finally:
        unlock_run(run_dir)

def upload_log(tid, method, model_path=None, echo=print, **ctx):
    # 업로드 한 건의 상세 기록 시작 → .say(메시지), .set(값=…), .finish(상태, 오류)
    #   tasks/<ID>/upload_<시각>.json·.log + logs/uploads.csv (한 줄씩 누적)
    mp = model_path or BEST
    mt, md5 = ctt.file_fingerprint(mp) if mp and os.path.exists(mp) else ('', '')
    return ctt.UploadLog(BASE, task_dir(tid), tid, method, echo=echo, server=CVAT_SERVER, profile=CVAT_LOGIN_MODE,
                         cvat_user=str(CVAT_SERVER).split('@')[0], operator=who_am_i(), session=SESSION_ID,
                         app_version=APP_VERSION, model=os.path.relpath(mp, BASE) if mp else '',
                         model_time=mt, model_md5=md5, **ctx)

def track_settings_of(xml_rel):
    # 5번 셀이 track을 만들 때 남긴 설정 (track_settings.json). 없으면 {}
    p = os.path.join(os.path.dirname(f'{BASE}/{xml_rel}'), 'track_settings.json')
    try:
        return json.load(open(p, encoding='utf-8'))
    except Exception:
        return {}

def change_report(task, before_zip, tag='after', show=True, max_frames=40):
    # 업로드 뒤 라벨을 export해 업로드 전 백업과 비교 → 라벨별·프레임별 '기존→새' 박스 수. CSV는 tasks/<ID>/change_<시각>.csv
    import pandas as pd
    from IPython.display import display
    folder = task_dir(task.id)
    after = ctt.export_annotations(task, f'{folder}/{tag}_{ctt.stamp()}.zip')
    rows, frows, text = ctt.label_change_report(before_zip, after)
    csv_path = ctt.save_change_report(rows, frows, f'{folder}/change_{ctt.stamp()}.csv')
    res = {'text': text, 'rows': rows, 'frames': frows, 'csv': os.path.relpath(csv_path, BASE), 'after': os.path.relpath(after, BASE)}
    if show:
        show_change(task.id, res, max_frames)
    return res

def show_change(tid, res, max_frames=40):
    import pandas as pd
    from IPython.display import display
    print(f"■ TASK {tid} 라벨 변화: {res['text']}")
    display(pd.DataFrame(res['rows']).set_index('라벨'))
    fr = res['frames']
    if fr:
        print(f"프레임별 박스 수 (기존→새){'' if len(fr) <= max_frames else f' — {len(fr)}프레임 중 앞 {max_frames}개, 전체는 CSV'}:")
        display(pd.DataFrame(fr[:max_frames]).set_index('프레임'))
    print(f"   저장: {res['csv']} (라벨별·프레임별 전체) · 업로드 후 라벨 {res['after']}")

_REQ = globals().get('_REQ') or {'next': None}   # 이 세션에서 받은 웹 요청 → 다음 작업 기록(job)에 묶임

def req_use(kind):
    # 웹 요청 받기 (셀의 FROM_REQUESTS = True): 대기 요청을 '진행 중'으로 바꾸고 TASK ID 목록(학습은 설정 dict)을 돌려줌.
    # 그 셀의 작업 기록이 끝날 때(job_end) 요청이 '완료'/'실패'로 바뀜 → 웹 화면 '요청' 탭에 보임
    if wb is None:
        raise SystemExit('⛔ scripts/web_bridge.py가 없습니다 (v4.14 scripts로 바꾸세요)')
    reqs = wb.take(BASE, kind, who_am_i(), SESSION_ID, limit=1 if kind in ('train', 'cleanup') else None,
                   **({'host': globals().get('REQ_HOST') or ('Kaggle' if ON_KAGGLE else 'Colab')} if 'host' in wb.take.__code__.co_varnames else {}))
    if not reqs:
        raise SystemExit(f"⛔ 받을 웹 요청이 없습니다 ({wb.KINDS[kind][0]}) → 웹 화면에서 요청하거나 FROM_REQUESTS = False")
    _REQ['next'] = (kind, [r['id'] for r in reqs])
    _REQ['params'] = dict(reqs[0].get('params') or {})   # 요청의 설정 (export 형식, 학습 규칙 등)
    for r in reqs:
        print(f"📨 웹 요청 {r['id']} · {wb.KINDS[kind][0]} · {r.get('by', '')} · TASK {len(r.get('task_ids') or [])}개"
              + (f" · {r['note']}" if r.get('note') else ''))
    _SCOPE_TMP.clear()
    if kind != 'train' and _REQ['params'].get('org') is not None:      # 웹에서 고른 TASK들의 조직으로 (프로젝트는 안 따짐) — v4.19
        _SCOPE_TMP.update(org=_REQ['params']['org'], project='')
        print(f"   → 조직: {_REQ['params']['org'] or '개인 작업공간'} (웹 요청의 TASK 조직, 이 작업이 끝나면 0번 셀 기본값으로)")
    if kind in ('train', 'cleanup'):                 # 설정 dict (모델 정리는 v4.24.8)
        return dict(reqs[0].get('params') or {})
    ids = wb.task_ids_of(reqs)
    print(f"   → TASK {len(ids)}개: {ids[:10]}{' …' if len(ids) > 10 else ''}")
    return ids

def _req_bind(job, action):
    nxt = _REQ.get('next')
    if job is not None and nxt and wb is not None and wb.KINDS[nxt[0]][2] == action:
        job['req'], _REQ['next'] = nxt, None

def _req_finish(job, status, summary):
    req = (job or {}).get('req')
    if req:
        _SCOPE_TMP.clear()                       # 웹 요청의 조직 → 다시 0번 셀 기본값
    if req and wb is not None:
        ok = status in ('끝', '완료')
        try:
            done = wb.finish(BASE, req[1], ok=ok, summary=summary or status, job_id=job.get('job_id', ''), who=who_am_i(),
                             **({'session': SESSION_ID} if 'session' in wb.finish.__code__.co_varnames else {}))   # v4.32.0: 이 세션이 받은 것만
            _w = sum(1 for d in done if d.get('status') == 'waiting')
            print(f"📨 웹 요청 {len(done)}건 → " + ('실패' if not ok else '완료' if not _w else
                  f"완료 {len(done) - _w} · 받는 중 {_w} (CVAT가 아직 준비 중 → 다 받으면 완료)") + " (웹 화면 '요청' 탭)")
        except Exception as e:
            print(f'(웹 요청 완료 표시 실패: {e})')

def job_start(cell, action, settings=None, summary=''):
    # 시계열 작업 기록(logs/activity.csv)에 작업 시작 — 이 셀의 TASK별 결과가 이 작업에 묶임 (26번 셀로 보기)
    if ctt is not None and hasattr(ctt, 'start_job'):
        try:
            _j = ctt.start_job(BASE, cell, action, who=who_am_i(), session=SESSION_ID,
                               server=globals().get('CVAT_SERVER', ''), model=model_label() if os.path.exists(BEST) else '',
                               settings=settings, summary=summary)
            _req_bind(_j, action)
            return _j
        except Exception as e:
            print(f'(작업 기록 시작 실패: {e})')

def job_end(status='끝', summary='', **kw):
    if ctt is not None and hasattr(ctt, 'current_job') and ctt.current_job():
        try:
            _j = ctt.current_job()
            ctt.end_job(status, summary, **kw)
            _req_finish(_j, status, summary)
        except Exception as e:
            print(f'(작업 기록 끝 실패: {e})')

def lock_status(name=None):
    # 학습 잠금 상태 한 줄
    d = ctt.read_run_lock(f'{BASE}/runs/{name or NAME}') if ctt else None
    if not d:
        return '없음'
    mine = ' (이 세션)' if d.get('session') == SESSION_ID else ''
    return (f"{d.get('who')}{mine} · 단계 {d.get('stage')} {d.get('epoch', '')} · 마지막 갱신 {d['age_min']:.0f}분 전"
            + ('  ← 오래됨(끊긴 학습)' if d['age_min'] >= LOCK_STALE_MIN else ''))

if ctt is not None and hasattr(ctt, 'import_old_records'):   # 처음 한 번: 예전 기록을 작업 기록(logs/activity.csv)으로
    try:
        ctt.import_old_records(BASE)
    except Exception as _e:
        print(f'(예전 기록 옮기기 실패: {_e})')
# ---------------- v4.31.5: ULD(CVAT_UpLoader) 폴더 읽기 · 라벨 업로드 뒤 이름 규칙 ----------------
# ULD 노트북이 내 드라이브 최상위 CVAT_UpLoader/ 에 남긴 기록(registry/tasks/<TASK ID>.json)과 clip(clips/…mp4)을 읽기만 함.
# 가지 앱은 그 폴더에 아무것도 쓰지 않음. 받은 파일은 작업 폴더 밖(임시 폴더)에 두어 eggplant_yolo/로 올라가지 않음.
ULD_REMOTE = globals().get('ULD_REMOTE') or 'gdrive,root_folder_id=root:CVAT_UpLoader'   # Kaggle: 같은 rclone 설정, 시작 폴더만 내 드라이브 최상위로
ULD_COLAB = globals().get('ULD_COLAB') or '/content/drive/MyDrive/CVAT_UpLoader'         # Colab: Drive 경로 그대로
ULD_DIR = f'{LOCAL_TMP}/_uld' if ON_KAGGLE else ULD_COLAB
_ULD = globals().get('_ULD') or {'rec': {}, 'media': None, 'media_t': 0}

def uld_pull(*rels, quiet=True):
    # ULD 폴더 → Kaggle 임시 폴더 (받기만 함). Colab에서는 Drive를 바로 읽으므로 아무것도 안 함
    if not ON_KAGGLE or not rels:
        return
    flt = [x for r in rels for x in ('--include', '/' + r.strip('/'))]
    _kg_rc(['copy', ULD_REMOTE, ULD_DIR, '--update', *flt], quiet=quiet)

def uld_task(tid, pull=True):
    # ULD가 만든 TASK면 그 기록(dict), 아니면 None (한 번 읽은 것은 기억)
    tid = str(tid)
    if tid in _ULD['rec']:
        return _ULD['rec'][tid]
    if pull:
        uld_pull(f'registry/tasks/{tid}.json')
    p = f'{ULD_DIR}/registry/tasks/{tid}.json'
    try:
        rec = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else None
    except Exception as e:
        print(f'⚠️ ULD 기록을 읽지 못함 ({tid}): {e}')
        rec = None
    _ULD['rec'][tid] = rec
    return rec

def uld_is_images(rec):
    return bool(rec) and rec.get('upload_as') == 'images' and bool(rec.get('frame_map'))

def uld_index(max_age=600):
    # v4.31.6: ULD가 만든 TASK {TASK ID: 'images'(사진 TASK) | 'video'} — 두 앱의 '사진(ULD)'·'ULD clip' 표시와 백업 판단용 (10분마다 새로 읽음)
    # v4.31.9 (F15): 읽기·교체 연결은 scripts의 wb.uld_scan (CPU 마스터와 같음) — 교체는 ULD가 '바꾸기'로 만든 기록만, 시각은 시간대를 맞춰 비교
    if _ULD.get('idx') is not None and time.time() - _ULD.get('idx_t', 0) < max_age:
        return _ULD['idx']
    sc = {'index': {}, 'lineage': {}, 'recs': {}}
    try:
        uld_pull('registry/tasks/*.json')
        sc = wb.uld_scan(f'{ULD_DIR}/registry/tasks')
        _ULD['rec'].update(sc['recs'])
    except Exception as e:
        print(f'(ULD 기록 목록을 읽지 못함 — 앱에 ULD 표시만 빠짐: {e})')
    _ULD['idx'], _ULD['idx_t'], _ULD['lineage'] = sc['index'], time.time(), sc['lineage']
    return sc['index']

def uld_lineage():
    # ULD가 바꿔 만든 TASK {옛 번호: 새 번호} (v4.31.8)
    uld_index()
    return dict(_ULD.get('lineage') or {})

def uld_video(tid):
    # ULD가 '영상'으로 올린 TASK의 원본 = ULD clip (v4.31.8) → clip 경로 또는 None (사진 TASK는 uld_clip + frame_map)
    rec = uld_task(tid)
    if not rec or uld_is_images(rec):
        return None
    return uld_clip(rec)

def uld_clip(rec):
    # 그 TASK의 clip 경로 (Kaggle은 받아 옴). 없거나 크기가 기록과 다르면 None
    rel = str(rec.get('clip_path') or '').split('CVAT_UpLoader/', 1)[-1]
    if not rel:
        return None
    uld_pull(rel, quiet=False)
    p = f'{ULD_DIR}/{rel}'
    if not os.path.exists(p):
        return None
    if rec.get('clip_bytes') not in (None, '') and os.path.getsize(p) != int(rec['clip_bytes']):
        print(f"⚠️ ULD clip 크기가 기록과 다름: {os.path.getsize(p)} ≠ {rec['clip_bytes']} ({rel})")
        return None
    return p

def uld_clip_done(path):
    # Kaggle: 다 쓴 clip을 지워 디스크를 아낌 (Colab은 Drive의 원본이므로 절대 지우지 않음)
    if ON_KAGGLE and path and str(path).startswith(ULD_DIR + '/') and os.path.exists(path):
        os.remove(path)

def uld_mismatch(task, rec):
    # 사진 TASK와 ULD 기록이 맞는지: 사진 수 · 사진 이름 끝(_f<프레임 6자리>) · 사진 크기. 맞으면 '' , 아니면 이유
    fm = list(rec.get('frame_map') or [])
    try:
        frames = list(task.get_meta().frames)
    except Exception as e:
        return f'CVAT 사진 목록을 읽지 못함: {ctt.api_error(e) if hasattr(ctt, "api_error") else e}'
    if len(frames) != len(fm):
        return f'사진 수 {len(frames)} ≠ ULD frame_map {len(fm)}'
    for k, fr in enumerate(frames):
        stem = os.path.splitext(str(fr.name))[0]
        if not stem.endswith(f'_f{int(fm[k]):06d}'):
            return f'사진 {k}번 이름 {fr.name} ≠ _f{int(fm[k]):06d} (ULD frame_map)'
        if rec.get('clip_w') and rec.get('clip_h') and (int(fr.width), int(fr.height)) != (int(rec['clip_w']), int(rec['clip_h'])):
            return f"사진 크기 {fr.width}x{fr.height} ≠ clip {rec['clip_w']}x{rec['clip_h']}"
    return ''

def uld_images(tid, out_dir, names=None):
    # 사진 TASK의 사진을 clip에서 다시 뽑아 out_dir에 CVAT 사진 이름으로 저장 (15번 셀 학습 데이터 — 라벨만 받은 export용)
    # names = CVAT 사진 이름 목록(순서대로). 없으면 '<만들 때 이름>_f<프레임 6자리>.jpg'. 반환: 새로 저장한 장 수 (ULD 사진 TASK가 아니면 None)
    rec = uld_task(tid)
    if not uld_is_images(rec):
        return None
    fm = [int(x) for x in rec['frame_map']]
    names = list(names) if names else [f"{rec.get('task_name_initial', '')}_f{n:06d}.jpg" for n in fm]
    os.makedirs(out_dir, exist_ok=True)
    todo = {n: nm for n, nm in zip(fm, names) if not os.path.exists(f'{out_dir}/{nm}')}
    if not todo:
        return 0
    clip = uld_clip(rec)
    if clip is None:
        print(f'⚠️ ULD clip을 받지 못해 TASK {tid}의 사진을 만들지 못함')
        return 0
    import av
    n_saved, last = 0, max(todo)
    with av.open(clip) as c:
        st = c.streams.video[0]
        for i, fr in enumerate(c.decode(st)):
            if i in todo:
                img = ctt.upright_image(fr.to_image(), ctt.pyav_rotation(fr, st)) if hasattr(ctt, 'upright_image') else fr.to_image()
                img.save(f'{out_dir}/{todo[i]}', quality=95)
                n_saved += 1
            if i >= last:
                break
    uld_clip_done(clip)
    return n_saved

def has_source_video(task):
    # 이 TASK의 원본 영상이 확보돼 있나: Drive videos/ · TASK 백업 · ULD clip (v4.31.5 — 없을 때만 백업 요청)
    name, tid = task.name, task.id
    try:
        if uld_task(tid):
            return True
    except Exception:
        pass
    try:
        if _ULD['media'] is None or time.time() - _ULD['media_t'] > 600:
            if ON_KAGGLE:
                _ls = lambda sub: [l for l in _kg_rc(['lsf', '-R', '--files-only', _kg_remote(sub)], quiet=True).stdout.splitlines() if l]
                _ULD['media'] = wb.media_names(BASE, remote_videos=_ls('videos'), remote_backups=_ls('backups'))
            else:
                _ULD['media'] = wb.media_names(BASE)
            _ULD['media_t'] = time.time()
        vn, bn = _ULD['media']
        b = ctt.base_name(name)
        return b.lower() in vn or b in bn
    except Exception as e:
        print(f'(영상 확보 여부를 확인하지 못함: {e})')
        return True                                  # 모르면 요청하지 않음

AA_REPLACED_NOTE = '_AA의 annotation export, backup하고 새로운 annotation으로 교체함'

def label_uploaded(task, replaced=False, before_zip=None, say=print, rename=True):
    # 라벨을 CVAT에 올린 뒤 (v4.31.5 이름 규칙 — 5·6·10·27·30번 셀 공통)
    #   …_uld → …_uld_AA 로 이름을 바꿈 · …_AA는 이름 그대로, 기존 라벨을 바꿨으면 그 사실을 기록:
    #   올리기 전 라벨(before_zip)을 exports/_aa_replaced/에 남기고, 원본 영상이 없으면(videos/·백업·ULD clip) 백업 요청
    # 반환 {'name': 지금 이름, 'renamed': bool, 'note': 기록에 붙일 말}
    old = task.name
    out = {'name': old, 'renamed': False, 'note': ''}
    new = ctt.aa_name_after_upload(old) if rename else old
    if new != old:
        try:
            task.update({'name': new})
            out.update(name=new, renamed=True, note=f'이름 변경 {old} → {new}')
            say(f'이름 변경: {old} → {new}')
        except Exception as e:
            out['note'] = f'이름 변경 실패 {ctt.api_error(e)}'
            say(f'⚠️ 이름 변경 실패 (라벨은 올라감): {ctt.api_error(e)}')
        return out
    if ctt.is_aa_name(old) and replaced:
        notes = [AA_REPLACED_NOTE]
        try:
            if before_zip and os.path.exists(str(before_zip)):
                d = f'{BASE}/exports/_aa_replaced'
                os.makedirs(d, exist_ok=True)
                dst = f"{d}/{ctt.export_file_stem(old)}__{task.id}__{ctt.stamp()}.zip"
                shutil.copy2(str(before_zip), dst)
                notes.append(f'이전 라벨 {os.path.relpath(dst, BASE)}')
        except Exception as e:
            notes.append(f'이전 라벨 보관 실패 {e}')
        try:
            if not has_source_video(task):
                r = wb.new_request(BASE, 'backup', [task.id], {'mode': 'all', 'export_reviewed': False, 'auto': 'aa_replace'},
                                   by=f'{who_am_i()} (AA 교체)', note='AA 라벨 교체 — 원본 영상 없음', task_names=[old])
                if r:
                    notes.append(f"영상 없음 → 백업 요청 {r['id']}")
        except Exception as e:
            notes.append(f'백업 요청 실패 {e}')
        out['note'] = ' · '.join(notes)
        say(out['note'])
    return out

if ON_KAGGLE:
    _free = shutil.disk_usage(BASE).free / 2**30
    print(f"실행 환경  : Kaggle ({'Save & Run All' if KAGGLE_BATCH else '대화형'}) · 작업 폴더 {BASE} ↔ Drive {_kg_remote() or KAGGLE_REMOTE} · "
          f"디스크 여유 {_free:.0f} GB · 자동 동기화 {'켬' if KAGGLE_AUTO_SYNC else '끔 → drive_sync_now()'}")
else:
    print('실행 환경  : Colab (Drive 연결)')
print(f'GPU        : {torch.cuda.get_device_name(0) if torch.cuda.is_available() else ("없음 (오른쪽 Session options → Accelerator → GPU)" if ON_KAGGLE else "없음 (런타임 유형을 T4 GPU로)")}'
      + (f' × {torch.cuda.device_count()} (15번 셀 학습은 모두 씀 — TRAIN_GPUS)' if torch.cuda.device_count() > 1 else ''))
print(f'패키지     : ultralytics {ultralytics.__version__}, cvat-sdk {CVAT_SDK_VER}, cvat-cli {CVAT_CLI_VER}')
print(f'CVAT 서버  : {SERVER_VER or "버전 확인 불가"} → SDK 호환: {cvat_compat()}')
print(f"CVAT 범위  : 기본 조직 {'구분 안 함 (모든 조직)' if CVAT_ORG is None else (CVAT_ORG or '개인 작업공간')} · "
      f"기본 프로젝트 {CVAT_PROJECT or '전부'}" + (' — 셀이 CVAT에 접속할 때 확인' if CVAT_ORG is not None or CVAT_PROJECT else ''))
if DS:
    print(f'데이터셋   : {DS_SOURCE} → {DS} (train {len(os.listdir(f"{DS}/train/images"))} / val {len(os.listdir(f"{DS}/val/images"))})  ← 2·3번 셀이 쓰는 데이터')
else:
    print('데이터셋   : dataset.zip 없음 (학습을 안 할 거면 괜찮음)')
print(f'선택 모델  : {NAME} (MODEL_CHOICE={MODEL_CHOICE!r})' + ('' if os.path.isdir(RUN_DIR) else '  ⚠️ runs/에 이 폴더가 없습니다 (이름 확인 또는 15번 셀로 학습)'))
print(f'학습 상태  : {train_status()}  [{RUN_DIR}]')
print(f'학습 잠금  : {lock_status()}')
print(f'내 이름    : {who_am_i()} (세션 {SESSION_ID})' + ('' if OPERATOR else "  ← 0번 셀 OPERATOR에 이름을 적으면 잠금 안내에 표시됨"))
print(f'best.pt    : {"있음 — " + model_label() if os.path.exists(BEST) else "없음"}')
if _trained:
    import datetime as _dt
    print('학습된 모델 (최근순, 쓰려면 MODEL_CHOICE에 이름 또는 번호):')
    _num = {v: k for k, v in MODEL_NAMES.items()}
    for d in _trained[::-1][:8]:
        n = os.path.basename(os.path.dirname(d))
        print(f"   {'▶' if n == NAME else ' '} {n:<45} 완료 {_dt.datetime.fromtimestamp(os.path.getmtime(d)).strftime('%m-%d %H:%M')}"
              + (f'  (번호 {_num[n]})' if n in _num else ''))
print(f'videos/    : {sum(1 for f in Path(VIDEOS).rglob("*") if f.suffix.lower() in (".mp4", ".mov", ".avi", ".mkv"))}개 영상 (하위 폴더 포함)')
_m = cvat_mode()
for _k, _p in CVAT_PROFILES.items():
    print(f"CVAT {_k:<8}: {_p['host']} · {'PAT' if prof_login(_k) == 'token' else 'ID/PASS'} · 보안 비밀 "
          f"{'있음' if _has_secret(_k) else '없음(실행 때 입력창)'}" + ('  ← 처음 접속' if _k == _m else ''))
print(f"학습 데이터 : 검토 완료 = 상태에 '{REVIEWED_RULE}' · val = '{VAL_RULE or '무작위(VAL_RATIO)'}' · "
      f"TASK 학습 대장 {len(ctt.read_train_tasks(BASE)) if ctt is not None else 0}줄"
      + (f" · 기록 세션 파일 {ctt.SHARD['id']}" if ctt is not None and hasattr(ctt, 'SHARD') else ''))
if wb is not None:
    _open = [r for r in wb.list_requests(BASE) if r['status'] in ('pending', 'taken')]
    print(f"웹 요청    : 대기·진행 중 {len(_open)}건" + (' → 28번 셀에서 보기, 해당 셀 FROM_REQUESTS = True로 실행' if _open else ''))
print(f'작업 대장  : {REGISTRY if os.path.exists(REGISTRY) else "아직 없음 (첫 작업 때 생성)"}')
_need = ['cvat_task_tools.py', 'cvat_backup.py', 'cvat_to_yolo.py', 'yolo_track_to_cvat.py',
         'eval_occluded_recall.py', 'compare_tracks.py', 'cvat_yolo_function.py', 'diagnose_pipeline.py', 'drive_revisions.py', 'ensemble_annotate.py', 'web_bridge.py', 'relabel_bridge.py']
missing = [f for f in _need if not os.path.exists(f'{SCRIPTS}/{f}')]
def _file_ver(path):
    m = re.search(r'^__version__\s*=\s*"([^"]+)"', open(path, encoding='utf-8').read(), re.M)
    return m.group(1) if m else '버전 없음(예전 파일)'
_wrong = {f: _file_ver(f'{SCRIPTS}/{f}') for f in _need if f not in missing}
SCRIPTS_OK = (APP_VERSION,)                   # v4.31.0: scripts도 바뀜 (세션별 기록 — cvat_task_tools·web_bridge·cvat_backup) → scripts/를 4.30.1로
_wrong = {f: v for f, v in _wrong.items() if v not in SCRIPTS_OK}
_dups = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*(*).py'))
print(f'앱 버전    : {APP_VERSION}')
print('scripts/   : ' + ('모두 있음, 버전 일치' if not missing and not _wrong else
                         ('없음 → ' + ', '.join(missing) + ' ' if missing else '')
                         + ('⚠️ 버전 다름 → ' + ', '.join(f'{f}({v})' for f, v in _wrong.items())
                            + f'  ← Drive scripts/를 v{APP_VERSION} 파일로 바꾸고 세션 다시 시작' if _wrong else '')))
if _dups:
    print(f'⚠️ scripts/에 이름이 겹친 파일: {", ".join(_dups)} → 지우세요 (Drive가 같은 이름 업로드를 새 파일로 만든 것)')
if ON_KAGGLE:                                   # scripts/ 안에 Colab 전용 코드가 남아 있는지 (Kaggle에서 안 될 수 있음)
    _colab_only = sorted(os.path.basename(f) for f in glob.glob(f'{SCRIPTS}/*.py')
                         if os.path.basename(f) != 'drive_revisions.py'
                         and re.search(r'google\.colab|/content/', open(f, encoding='utf-8').read()))
    if _colab_only:
        print(f"⚠️ Colab 전용 코드가 있는 scripts: {', '.join(_colab_only)} → 이 파일을 쓰는 셀은 Kaggle에서 확인 필요")
    print('Kaggle     : 23번 셀(Drive 버전 복원)은 Colab에서만 · 영상·백업은 쓰는 셀이 Drive에서 자동으로 받음 · 직접: drive_pull(경로), drive_get_videos(검색어)')


In [ ]:
# 30. 작업자 — 두 앱의 요청을 한 셀에서 공정하게 처리 (v4.31.6: 예전 30번 재처리 + 31번 LC 작업자를 합침 · v4.31.7: 학습 요청 · GPU 워커)
#     · AA앱(가지 재처리): uld·AA TASK의 shape·track 라벨링 · track 올리기 · 백업 → 뒤에서 여러 줄로 동시에 (GPU 줄 · 입출력 줄)
#     · LC앱 '가지 라벨링 관리': 백업 · export · track 생성 · 자동 라벨 · 모델 정리 · 모델 비교 · 학습 → 그 셀(5·9·10·13·14·15·29·32번)을 차례로 실행
#     · v4.31.7: W_ROLE = 'gpu' = Kaggle GPU 워커 (CPU 마스터가 시작) — kaggle_gpu로 배정된 GPU 요청만 받고, 일이 없으면 스스로 끝나 GPU를 돌려줌
#     · 공정 차례: 가벼운 일(새로 고침 등) 먼저 → 사람마다 돌아가며 한 건씩(각자 가장 오래된 것부터) → 같은 차례 안에서는 오래 기다린 것 먼저.
#       밀린 요청이 많은 사람이 다른 사람의 새 요청을 막지 않음 · 한 사람이 무거운 일을 W_USER_MAX건 넘게 동시에 차지하지 않음 (다른 사람이 기다릴 때)
#     · GPU: LC의 track·자동 라벨·비교가 실행되는 동안 GPU 한 줄을 맡아 두고(학습은 모든 줄), 재처리 라벨링은 남은 줄로
#     · 요청 확인은 Drive를 W_POLL_SEC초마다 (일이 끝나면 바로) · 같은 TASK의 요청은 보낸 차례대로 하나씩
#     · 멈추려면 셀 중지(■) — 새 요청은 받지 않고, 하던 TASK까지 끝낸 뒤 멈춤 (남은 TASK는 요청에 '대기'로 남아 다음에 이어 받음)
assert 'BASE' in globals(), '먼저 0번 셀(공통 설정·준비)을 실행하세요'
W_LOGIN      = None     # CVAT 계정 (0번 셀 CVAT_PROFILES의 이름) · None = 지금 그대로
W_HOURS      = 8        # 이 시간 동안 켜 둠
W_POLL_SEC   = None     # 새 요청 확인 간격(초) · None = Colab 10초 · Kaggle 20초 (Kaggle은 매번 rclone으로 받음)
W_USER_MAX   = 2        # 한 사람이 동시에 차지하는 무거운 일(라벨링·올리기·백업) 수 — 다른 사람이 기다릴 때만 · 0 = 제한 없음
W_STATUS_MIN = 10       # 요청이 없을 때 이 간격(분)마다 '대기 중 · 열린 요청 n건' 한 줄
W_TAKEOVER   = False    # True = 다른 세션의 작업자가 켜져 있어도 넘겨받음
W_MAX_LOOPS  = None     # (시험용) 이만큼 돌고 멈춤
W_ROLE       = 'gpu'    # 'all' = 두 앱의 요청 모두 (Colab·Kaggle에서 사람이 켬) · 'gpu' = Kaggle GPU 워커 (CPU 마스터가 시작, kaggle_gpu 배정만)
W_CLAIM_CONFIRM_SEC = 2  # v4.32.0: 요청·취합자를 맡은 뒤 이만큼 기다렸다 다시 읽어 확인 (거의 동시에 맡은 다른 작업자가 있으면 물러남)
W_EXIT_IDLE_MIN = None  # 할 일이 없는 채로 이 시간(분)이 지나면 끝냄 · None = W_HOURS까지 켜 둠 ('gpu'이면 5분)
# ---- 재처리 (AA앱(가지 재처리))
RL_REFRESH_MIN = 15       # CVAT TASK 목록을 새로 읽는 간격 (분) — 폰의 '새로 고침'은 바로
RL_GPU_WORKERS = 2        # shape·track 라벨링을 동시에 몇 TASK까지 (LC의 GPU 일도 이 수 안에서) · CUDA out of memory면 줄이세요
RL_IO_WORKERS  = 2        # 올리기·백업을 동시에 몇 건까지 · CVAT 서버가 느려지면 줄이세요
RL_BACKUP_POLL_MIN = 1    # '받는 중' 백업을 CVAT에 확인하는 간격 (분)
RL_BACKUP_GIVEUP_H = 6    # 이 시간이 지나도 준비되지 않은 백업은 포기 (그 TASK는 실패로 → 다시 요청)
RL_REPLACE_ANY = False    # v4.31.9 (F5): True = 이름이 uld·AA가 아닌 TASK(사람이 검토해 _vf 등으로 바꾼 것)도 기존 라벨을 교체 · False = 보류 (권장)
# ---- LC (LC앱 '가지 라벨링 관리')
LC_KINDS        = ['cleanup', 'backup', 'export', 'aa', 'track', 'compare', 'train']   # 받을 종류 (v4.31.7: 학습도 요청으로 — 15번 셀) · 차례는 위 공정 차례
LC_GPU_KINDS    = ['track', 'aa', 'compare', 'train']   # GPU를 쓰는 LC 일 (실행 중 GPU 한 줄을 맡아 둠 · 학습은 모든 줄이 빈 뒤 시작)
LC_DOWNLOAD_MIN = 10      # '받는 중' 백업·export: CVAT가 준비한 것을 이 간격(분)마다 받음
LC_STATE_MIN    = 30      # LC앱 상태(web/state.json)를 28번 셀로 새로 만드는 간격(분) — 요청을 처리한 뒤에도 새로 만듦
LC_STALE_TAKEN_H = 6      # 다른 세션이 '진행 중'으로 둔 요청이 이 시간(시간)이 지나면 끊긴 것으로 보고 다시 대기로
# 각 LC 요청은 그 셀의 코드를 그대로(FROM_REQUESTS = True) 실행 — 셀 기본설정을 따름. 예: 자동 라벨은 9번 셀 AA_EXISTING = 'skip',
# track은 5번 셀 TRACK_AUTO_UPLOAD = False (노트북에서 셀을 고쳐도 작업자는 scripts/notebook_cells.json의 기본값으로 실행).
# 바꾸려면 아래 LC_SET에 '셀 번호': {'설정 이름': 값}을 적음.
LC_SET          = {}      # 예: {'5': {'TRACK_AUTO_UPLOAD': True}, '9': {'AA_EXISTING': 'replace'}, '13': {'BK_MAX_WAIT': 900}}

if W_POLL_SEC is None:
    W_POLL_SEC = 20 if ON_KAGGLE else 10
RL_LOGIN = LC_LOGIN = W_LOGIN
RL_HOURS = LC_HOURS = W_HOURS
RL_POLL_SEC = LC_POLL_SEC = W_POLL_SEC
RL_STATUS_MIN = LC_STATUS_MIN = W_STATUS_MIN
RL_TAKEOVER = LC_TAKEOVER = W_TAKEOVER
RL_MAX_LOOPS = LC_MAX_LOOPS = W_MAX_LOOPS
RL_STALE_TAKEN_H = LC_STALE_TAKEN_H
W_SLOT = '_gpu' if W_ROLE == 'gpu' else ''        # v4.31.7: 작업자 기록 파일 (GPU 워커는 web/worker_gpu.json · relabel/WORKER_gpu.json)
REQ_HOST = 'kaggle_gpu' if W_ROLE == 'gpu' else None   # 0번 셀 req_use(셀이 요청을 받을 때)의 배정 확인
if W_ROLE == 'gpu':
    LC_KINDS = [k for k in LC_KINDS if k in LC_GPU_KINDS]
    if W_EXIT_IDLE_MIN is None:
        W_EXIT_IDLE_MIN = 5
assert wb is not None and hasattr(wb, 'lease_should_take'), 'scripts/를 v4.32.0으로 바꾸세요'
from collections import Counter
_W = {'stop': False, 'lc_gpu': 0, 'aa_err': None, 'aa_done': False, 'served': [], 'idle_since': None}

def _w_mine(rid, host):
    # 이 작업자가 받을 요청인가 (배정 requests/assign) · GPU 워커는 배정 파일이 있는 것만 (사람이 켠 30번 셀 몫을 가져가지 않게)
    if W_ROLE == 'gpu' and not os.path.exists(f'{BASE}/requests/assign/ASSIGN_{rid}.json'):
        return False
    return wb._assigned_to_me(BASE, rid, host)
W_FAIR_MIN = 60          # 돌아가며: 최근 이 시간(분) 동안 차례를 받은 횟수만큼 그 사람을 뒤로

def _w_served():
    # 최근 W_FAIR_MIN분 동안 사람마다 맡긴 무거운 일 수 (재처리 + LC)
    t = _time.time() - W_FAIR_MIN * 60
    _W['served'] = [(x, u) for x, u in _W['served'] if x >= t]
    return Counter(u for _, u in _W['served'])

def _w_served_add(u):
    _W['served'].append((_time.time(), u))

# =============== 재처리 (예전 30번 셀) ===============
import glob, json, threading, time as _time, traceback, importlib
from concurrent.futures import ThreadPoolExecutor, wait as _fwait
import torch
import relabel_bridge as rb
importlib.reload(rb)

_old = globals().get('_RL_POOLS')                # 앞 실행을 ■로 멈췄을 때 아직 끝나지 않은 일 → 끝날 때까지 기다림 (같은 요청을 두 번 하지 않게)
if _old:
    _left = [f for f in list(_old['futs']) if not f.done()]
    if _left:
        print(f"⏳ 앞 실행에서 하던 일 {len(_left)}건이 끝나기를 기다립니다 …")
        _fwait(_left)
    for _p in ('gpu', 'io', 'task'):
        if _p in _old:
            _old[_p].shutdown(wait=False)
use_cvat(RL_LOGIN)
_RL = {'stop': False, 'heart_stop': False, 'started': ctt.now(), 'host': 'kaggle_gpu' if W_ROLE == 'gpu' else 'Kaggle' if ON_KAGGLE else 'Colab', 'refresh_t': 0,
       'bk_t': 0, 'state_t': 0, 'dirty': False, 'gpu': {}, 'io': {}, 'tasks': {}, 'ready': {}}
_RL_LOCK = threading.RLock()                    # 요청 파일·작업 대장·Drive 동기화 (줄끼리 겹치지 않게)
_RL_BK_LOCK = threading.Lock()                  # 백업 스크립트는 한 번에 하나 (설정·대기 목록을 같이 씀)
_RL_EV = threading.Event()                      # 일이 끝나면 바로 다음 요청을 보게
_RL_TL = threading.local()
_RL_POOLS = {'gpu': ThreadPoolExecutor(max(1, int(RL_GPU_WORKERS)), thread_name_prefix='rl-gpu'),   # 요청을 맡아 TASK를 나눠 주는 쪽
             'task': ThreadPoolExecutor(max(1, int(RL_GPU_WORKERS)), thread_name_prefix='rl-gput'),  # 라벨링 TASK를 실제로 하는 줄 (v4.29.3)
             'io': ThreadPoolExecutor(max(1, int(RL_IO_WORKERS)), thread_name_prefix='rl-io'), 'futs': {}}
_RL_FUNC = f'{SCRIPTS}/cvat_yolo_function.py'
_RL_NGPU = torch.cuda.device_count() if torch.cuda.is_available() else 0

def _rl_dev():
    # 이 줄이 쓸 GPU — GPU가 여럿(Kaggle T4×2)이면 줄마다 나눠 씀, 없으면 CPU (v4.29.3)
    if not _RL_NGPU:
        return 'cpu'
    try:
        i = int(threading.current_thread().name.rsplit('_', 1)[-1])
    except ValueError:
        i = 0
    return str(i % _RL_NGPU)
_RL_GPU, _RL_IO, _RL_LIGHT = ('shape', 'track'), ('upload', 'backup'), ('refresh', 'hide', 'unhide')
_RL_OPEN_BK = ('요청됨', '준비 중')
rb.ensure(BASE)
if ON_KAGGLE:
    drive_pull('relabel/**', quiet=True)
_RL_PREV = rb.acquire_worker(BASE, who_am_i(), SESSION_ID, takeover=RL_TAKEOVER, slot=W_SLOT) or {}   # 앞 작업자 (v4.31.4: 그 세션이 남긴 '진행 중'만 되돌림)

def _rl_push(*rels):
    # Kaggle: 바뀐 것을 Drive로 (Colab은 Drive에 바로 씀) · v4.32.0: 올릴 파일을 꼭 적음 (relabel/** 통째로 올리지 않음)
    if ON_KAGGLE and rels:
        drive_push(*rels, quiet=True)

# ---------- v4.32.0 (F3·F14): 공유 파일은 쓰는 사람을 하나로 ----------
# · 상태 파일(relabel/state.json·tracked.json, web/state.json …)은 취합자(jobs/STATE_OWNER.json 임대)만 — CPU 마스터(3) > 이 작업자(2), GPU 워커(0)는 맡지 않음
# · 요청 파일은 받은 세션만 (쓰기 직전 Drive에서 다시 확인) · '받는 중'은 받은 세션이 꺼졌을 때만 넘겨받음
# · TASK 기록은 최신본을 받아 고쳐 그 파일만 올림 (rb.sync_record — Kaggle) · Colab은 Drive에 바로 고침
W_STATE_PRIO = wb.LEASE_PRIO['gpu' if W_ROLE == 'gpu' else 'worker']
_OWN = {'mine': False, 't': 0, 'mark_t': 0}
_OWN_LOCK = threading.RLock()

def _kg_fetch(rel, dest):
    d, name = rel.rsplit('/', 1)
    ls = _kg_rc(['lsf', '--files-only', '--include', name, _kg_remote(d)], quiet=True)
    if ls.returncode != 0:
        return 'missing' if 'directory not found' in (ls.stderr or '').lower() else 'fail'
    if name not in {x.strip() for x in (ls.stdout or '').splitlines()}:
        return 'missing'
    r = _kg_rc(['copyto', _kg_remote(rel), str(dest)], quiet=True)
    return 'ok' if r.returncode == 0 and os.path.isfile(dest) else 'fail'

def _kg_put(rel, src):
    return _kg_rc(['copyto', str(src), _kg_remote(rel)], quiet=True).returncode == 0

if ON_KAGGLE:
    rb.use_remote(_kg_fetch, _kg_put)

def _lease_cur():
    # Drive의 취합자 임대 → ('ok'|'missing'|'fail', 내용)
    if ON_KAGGLE:
        tmp = f'{LOCAL_TMP}/_lease_{SESSION_ID}.json'
        st = _kg_fetch(wb.LEASE_FILE, tmp)
        if st != 'ok':
            return st, None
        try:
            return 'ok', json.load(open(tmp, encoding='utf-8'))
        except Exception:
            return 'fail', None
    return ('ok', wb.lease_read(BASE)) if os.path.exists(f'{BASE}/{wb.LEASE_FILE}') else ('missing', None)

def _lease_write(stopping=False):
    wb.lease_write(BASE, wb.lease_body(SESSION_ID, who_am_i(), W_ROLE, W_STATE_PRIO, stopping=stopping))
    return _kg_put(wb.LEASE_FILE, f'{BASE}/{wb.LEASE_FILE}') if ON_KAGGLE else True

def _own_state(force=False, stopping=False):
    # 이 작업자가 상태 파일 취합자인가 — 60초마다 임대를 다시 확인·갱신
    if W_STATE_PRIO <= 0:
        return False
    with _OWN_LOCK:
        if not (force or stopping) and _time.time() - _OWN['t'] < 60:
            return _OWN['mine']
        _OWN['t'] = _time.time()
        st, cur = _lease_cur()
        was = _OWN['mine']
        if st == 'fail':                           # Drive를 못 읽으면 맡지 않음 (두 명이 되지 않게)
            _OWN['mine'] = False
            return False
        mine_now = bool(cur) and cur.get('session') == SESSION_ID
        if stopping:
            if mine_now:
                _lease_write(stopping=True)
            _OWN['mine'] = False
            return False
        if not wb.lease_should_take(cur, SESSION_ID, W_STATE_PRIO) or not _lease_write():
            if was:
                print(f"🔑 [{ctt.now()[11:16]}] 상태 파일 취합자를 넘김 → {(cur or {}).get('who', '?')} ({(cur or {}).get('role', '')})")
            _OWN['mine'] = False
            return False
        if not mine_now:
            _time.sleep(W_CLAIM_CONFIRM_SEC)
            st2, again = _lease_cur()
            if not again or again.get('session') != SESSION_ID:
                _OWN['mine'] = False
                return False
            if ON_KAGGLE:                          # 앞 취합자가 쓴 tracked·상태를 받아 그 위에 이어 씀
                for _rel in ('relabel/tracked.json', 'relabel/state.json'):
                    if _kg_fetch(_rel, f'{BASE}/{_rel}.lease.tmp') == 'ok':
                        os.replace(f'{BASE}/{_rel}.lease.tmp', f'{BASE}/{_rel}')
            print(f"🔑 [{ctt.now()[11:16]}] 상태 파일 취합자를 맡음 (jobs/STATE_OWNER.json)")
        _OWN['mine'] = True
        return True

def _dirty_mark(why=''):
    # 취합자가 아닐 때: 상태 파일 대신 '새로 만들어 주세요' 표시만 (30초에 한 번)
    if _time.time() - _OWN['mark_t'] < 30:
        return
    _OWN['mark_t'] = _time.time()
    p = wb.dirty_mark(BASE, SESSION_ID, why=why)
    if ON_KAGGLE:
        _kg_put(f'jobs/{p.name}', p)

def _owner_marks():
    # 취합자: 다른 작업자가 남긴 표시(DIRTY_*·DONE_*) → 상태를 곧 새로 만들게 하고 표시는 지움
    if not _own_state():
        return 0
    if ON_KAGGLE:
        ls = _kg_rc(['lsf', '--files-only', '--include', 'DIRTY_*.json', '--include', 'DONE_*.json', _kg_remote('jobs')], quiet=True)
        names = [x.strip() for x in (ls.stdout or '').splitlines() if x.strip()] if ls.returncode == 0 else []
    else:
        names = [os.path.basename(p) for p in glob.glob(f'{BASE}/jobs/DIRTY_*.json') + glob.glob(f'{BASE}/jobs/DONE_*.json')]
    names = [n for n in names if SESSION_ID not in n]
    if names:
        _RL['refresh_t'] = 0                       # 다음 차례에 목록·상태를 새로
        _LC['dirty'] = True
        for n in names:
            if ON_KAGGLE:
                _kg_rc(['deletefile', _kg_remote(f'jobs/{n}')], quiet=True)
            try:
                os.remove(f'{BASE}/jobs/{n}')
            except OSError:
                pass
    return len(names)

def _req_remote(rid):
    # Drive에 지금 있는 그 AA 요청 (못 찾거나 둘 이상이면 None)
    if not ON_KAGGLE:
        return _rl_req(rid)
    ls = _kg_rc(['lsf', '--files-only', '--include', f'REQ_{rid}__*', _kg_remote('relabel/requests')], quiet=True)
    names = sorted(x.strip() for x in (ls.stdout or '').splitlines() if x.strip()) if ls.returncode == 0 else []
    if len(names) != 1:
        return None
    tmp = f'{LOCAL_TMP}/_req_{SESSION_ID}_{rid}.json'
    if _kg_fetch(f'relabel/requests/{names[0]}', tmp) != 'ok':
        return None
    try:
        cur = json.load(open(tmp, encoding='utf-8'))
        cur['_name'] = names[0]
        return cur
    except Exception:
        return None

def _rl_alive():
    # 지금 살아 있는 작업자 세션들 (생존 신호 파일)
    if ON_KAGGLE:
        drive_pull('web/worker*.json', 'relabel/WORKER*.json', quiet=True)
    return wb.live_sessions(BASE) | {SESSION_ID}

def _rl_cl():
    # 줄(스레드)마다 따로 CVAT 접속 — 동시에 써도 섞이지 않게 (주 스레드는 CLIENT)
    if threading.current_thread() is threading.main_thread():
        return CLIENT
    if getattr(_RL_TL, 'cl', None) is None:
        _RL_TL.cl = cvat_client(CVAT_LOGIN_MODE)
    return _RL_TL.cl

def _rl_reg(**row):
    with _RL_LOCK:
        ctt.append_registry(REGISTRY, **row)

def _rl_busy():
    # 폰에 보일 '지금 하는 일': GPU 줄 + 입출력 줄
    g, io = [dict(v) for v in list(_RL['gpu'].values())], [dict(v) for v in list(_RL['io'].values())]
    top = dict(g[0]) if g else dict(io[0]) if io else {'what': '대기 중'}
    if _RL.get('closing'):                     # ■ 뒤 하던 일만 마무리 — 새 요청은 받지 않음 (v4.29.4)
        top = {'what': f"멈추는 중 · 하던 일 {len(g) + len(io)}건 마무리", 'closing': True}
    top['lanes'] = {'gpu': g[0] if g else None, 'gpus': g, 'io': io}
    top['loop_at'] = _RL.get('loop_at')          # 요청 확인 루프가 마지막으로 돈 시각 (v4.29.4)
    return top

def _rl_beat(what=None, stopping=False):
    b = {'what': what} if what else _rl_busy()
    rb.worker_beat(BASE, who_am_i(), SESSION_ID, _RL['host'], busy=b, started=_RL['started'], stopping=stopping, slot=W_SLOT)
    _rl_push(f'relabel/WORKER{W_SLOT}.json')

def _rl_lane(info):
    # 이 줄이 지금 하는 일 (None = 끝)
    me = threading.current_thread().name
    lane = _RL['gpu'] if me.startswith('rl-gpu') else _RL['io']
    if info is None:
        lane.pop(me, None)
    else:
        lane[me] = info
    try:
        _rl_beat()
    except Exception:
        pass

def _rl_heart():
    while not _RL['heart_stop']:                 # 오래 걸리는 TASK 중에도 폰에 '켜짐'으로 보이게
        try:
            _rl_beat()
        except Exception:
            pass
        for _ in range(60):
            if _RL['heart_stop']:
                return
            _time.sleep(1)

def _rl_status(req, status, _adopt=False, **f):
    with _RL_LOCK:                               # 이름 바꾸기 → Drive에 올리기 → 예전 이름 지우기를 한 번에 (요청 받아 오기와 겹치지 않게)
        cur = _req_remote(req['id'])               # v4.32.0 (F3·F14): 쓰기 직전 Drive에서 아직 이 세션 것인지 확인
        ok_ = (wb.may_adopt(cur, SESSION_ID, _rl_alive()) and cur.get('status') == req.get('status')) if (_adopt and cur) else wb.request_owned(cur, req, SESSION_ID)
        if not ok_:
            print(f"ℹ️ 요청 {req['id']}: Drive에서 이 세션 것이 아님(다른 작업자가 받았거나 끝냄·취소) → 쓰지 않음")
            return dict(req, _skipped=True)
        old = req.get('_path')
        new = rb.set_status(BASE, req, status, by=who_am_i(), **f)
        if ON_KAGGLE:                            # Drive에 남은 예전 상태 파일 지움 (폰이 같은 요청을 둘로 보지 않게)
            drive_push('relabel/requests/' + os.path.basename(new['_path']), quiet=True)
            if old and os.path.basename(old) != os.path.basename(new['_path']):
                _kg_rc(['deletefile', _kg_remote('relabel/requests/' + os.path.basename(old))], quiet=True)
        return new

def _rl_req(rid):
    # 요청 파일을 다시 읽음 (그 사이 상태가 바뀌었을 수 있음)
    return next((r for r in rb.list_requests(BASE) if r['id'] == rid), None)

def _rl_model(name):
    if not name:
        raise RuntimeError('요청에 모델이 없습니다')
    p = f'{BASE}/runs/{name}/weights/best.pt'
    if ON_KAGGLE and not os.path.exists(p):
        drive_pull(f'runs/{name}/weights/best.pt', f'runs/{name}/*', quiet=True)
    if not os.path.exists(p):
        raise RuntimeError(f'모델 없음: runs/{name}/weights/best.pt')
    import datetime as _dt
    return p, f"{name}/best.pt ({_dt.datetime.fromtimestamp(os.path.getmtime(p)).strftime('%m-%d %H:%M')})"

def _rl_media():
    if ON_KAGGLE:
        _KG['ls'].pop('videos', None); _KG['ls'].pop('backups', None)
        return wb.media_names(BASE, remote_videos=[p for p, s, t in _kg_files('videos')],
                              remote_backups=[p for p, s, t in _kg_files('backups')])
    return wb.media_names(BASE, videos_dir=VIDEOS)

def _rl_refresh(force=False):
    if not force and _time.time() - _RL['refresh_t'] < RL_REFRESH_MIN * 60:
        return
    tasks = ctt.list_tasks(_rl_cl(), limit=5000)          # v4.31.6: 뒤 스레드에서도 (주 스레드는 LC 셀이 CLIENT를 씀)
    if ON_KAGGLE:
        drive_pull('status/cvat_tasks.json', quiet=True)   # v4.31.8: 사라진 TASK (수집기 기록)
    _RL['media'] = _rl_media()
    _RL['models'], _RL['models_t'] = rb.model_rows(BASE, files=kg_file_index('runs') if ON_KAGGLE else None), _time.time()   # v4.31.6: Kaggle은 Drive 목록으로
    if not _own_state():                         # v4.32.0 (F3·F14): tracked·상태는 취합자만 — 아니면 표시만 남김
        _RL['refresh_t'] = _time.time()
        _RL['dirty'] = False
        _dirty_mark('목록 새로 고침')
        return
    tr = rb.update_tracked(BASE, tasks)
    st = rb.build_state(BASE, tr, models=_RL['models'], media=_RL['media'], uld=uld_index(), replaced=uld_lineage(), server=CVAT_SERVER, scope_text=ctt.scope_text(CVAT_SCOPE),
                        defaults={'conf': AA_CONF, 'imgsz': IMGSZ, 'model': globals().get('NAME', '')})
    _RL['refresh_t'] = _RL['state_t'] = _time.time()
    _RL['dirty'] = False
    _rl_push('relabel/state.json', 'relabel/tracked.json')
    s = st['summary']
    print(f"[{ctt.now()[11:16]}] 목록: 대상 {s['targets']} · 진척 {s['progressed']} · 숨김 {s['hidden']} · 확인 대기 track {s['pending_track']} · 모델 {len(st['models'])}개")

def _rl_state(force=False):
    # 폰 화면 상태 다시 쓰기 — 줄들이 바꾼 것을 모아 몇 초에 한 번 (주 스레드만)
    if not (force or _RL['dirty']) or (not force and _time.time() - _RL['state_t'] < 5):
        return
    _RL['dirty'] = False
    if not _own_state():                         # v4.32.0 (F3·F14): 상태 파일은 취합자만 — 아니면 표시만
        _RL['state_t'] = _time.time()
        _dirty_mark('재처리 작업 뒤')
        return
    if _RL.get('models') is not None and (_RL.get('closing') or _time.time() - _RL.get('models_t', 0) < RL_REFRESH_MIN * 60):
        _models = _RL['models']                    # v4.31.6: 모델 목록은 목록 새로 고침(RL_REFRESH_MIN) 때만 새로 — 상태를 쓸 때마다 만들지 않음
    else:
        _models = _RL['models'] = rb.model_rows(BASE, files=kg_file_index('runs') if ON_KAGGLE else None)
        _RL['models_t'] = _time.time()
    rb.build_state(BASE, models=_models, media=_RL.get('media') or _rl_media(), uld=uld_index(), replaced=uld_lineage(), server=CVAT_SERVER, scope_text=ctt.scope_text(CVAT_SCOPE),
                   defaults={'conf': AA_CONF, 'imgsz': IMGSZ, 'model': globals().get('NAME', '')})
    _RL['state_t'] = _time.time()
    _rl_push('relabel/state.json')

def _rl_video(task, name, source=''):
    v = ctt.find_video(VIDEOS, source, name)
    if v is None:
        v = ctt.video_from_backups(f'{BASE}/backups', name, VIDEOS)[0]
    return v

def _rl_backup(ids, mode):
    # 13번 셀과 같은 백업 (TASK 영상 + 라벨) · mode 'request' = 요청만 / 'download' = 준비된 것 받기 (한 번 확인) — 기다리지 않음
    with _RL_BK_LOCK:
        cb = load_script('cvat_backup.py')
        prof = CVAT_PROFILES[CVAT_LOGIN_MODE]
        cb.HOST, cb.AUTH_MODE, cb.BASE = CVAT_HOST, prof_login(CVAT_LOGIN_MODE), Path(BASE)
        cb.CREDENTIALS = cvat_auth(CVAT_LOGIN_MODE)[1]
        cb.ORGANIZATION_SLUG, cb.PROJECT_ID = CVAT_SCOPE.get('org'), CVAT_SCOPE.get('project_id')
        cb.NAME_QUERY, cb.TASK_IDS, cb.PREVIEW = '', list(ids), False
        cb.FORCE, cb.UPDATE_BOARD, cb.BOARD_MODE, cb.ONLY_STATES = False, True, 'file', None
        cb.MODE, cb.MAX_WAIT = mode, 0
        st = cb.run_backup() or []
        _RL['bk_rows'] = [r for r in st if str(r.get('task_id')) in {str(t) for t in ids}]   # v4.31.10 (R1): TASK별 판정용
        _RL['media'] = _rl_media()               # 받은 백업 → 영상을 얻을 곳
        _rl_push('backups/**', 'backup_index.csv', 'backup_status.csv', 'reports/**')
    want = {str(t) for t in ids}                  # 백업 스크립트는 작업보드용으로 모든 TASK의 백업 상태를 돌려줌 → 요청한 TASK만 (v4.27)
    return {str(r.get('task_id')): r.get('backup_state', '') for r in st if str(r.get('task_id')) in want}

def _rl_bk_left(ids, st):
    # 아직 받지 못한 TASK: 이 서버의 백업 요청이 열려 있음(요청됨·준비 중)
    op = {str(r.get('task_id')) for r in wb.pending_rows(BASE, 'backup')
          if r.get('status') in _RL_OPEN_BK and r.get('server', '') in ('', CVAT_SERVER)}
    return [t for t in ids if str(t) in op or st.get(str(t)) in _RL_OPEN_BK]

def _rl_clear_stale(tid, req=None):
    # shape로 CVAT 라벨이 바뀌면 그 전에 만든 track(확인 대기)은 지난 것 → 올리지 않게 정리 (파일은 남음) — v4.29
    rec = rb.task_record(BASE, tid)
    p = rec.get('track_pending')
    if not p:
        return False
    rb.set_pending_track(BASE, tid, None)
    rb.add_history(BASE, tid, kind='discard', status='지난 track 정리', request=(req or {}).get('id', ''), who=who_am_i(),
                   note=f"shape로 라벨이 바뀌어 {str(p.get('made', ''))[:16]}에 만든 track({p.get('tracks')}개)은 올리지 않음 · 파일 {p.get('xml')}")
    print(f'   [{tid}] 지난 track 확인 대기 정리 (shape로 라벨이 바뀜)')
    return True

def _rl_shape_one(tid, mpath, mlabel, conf, req):
    # 10번 셀과 같은 순서: 백업 → (기존 라벨 있으면 교체) 자동 라벨 → 전·후 비교 → 상세 로그 · 이름은 바꾸지 않음
    folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
    ts = ctt.stamp()
    row = dict(run_time=ctt.now(), action='auto', task_id=tid, model=mlabel, server=CVAT_SERVER)
    L, err, res = None, None, {}
    try:
        task = _rl_cl().tasks.retrieve(tid)
        row['task_name'] = task.name
        vid, off = None, (TRACK_FRAME_OFFSET if globals().get('AA_VIDEO_OFFSET') is None else AA_VIDEO_OFFSET)
        if ctt.task_basic(task)['is_video']:
            v = _rl_video(task, task.name)
            try:                                 # 9번 셀과 같은 확인: 영상과 CVAT 프레임이 같은 장면이면 원본 화질로, 아니면 CVAT 프레임
                if v is not None and ctt.check_video_alignment(task, v, off, 'pyav')['ok']:
                    vid = v
            except Exception as _e:
                print(f'   [{tid}] (영상 맞춤 확인 실패 → CVAT 프레임: {ctt.api_error(_e)})')
        L = upload_log(tid, 'shape', model_path=mpath, echo=lambda m: print(f'   [{tid}] {m}'), task_name=task.name,
                       source=(f'원본 영상 {os.path.relpath(str(vid), BASE)} (pyav, 보정 {off:+d})' if vid else 'CVAT 프레임 (original)'),
                       settings={'imgsz': IMGSZ, 'conf': conf, 'existing': 'replace', 'app': 'relabel', 'request': req['id']})
        backup = ctt.export_annotations(task, f'{folder}/before_auto_{ts}.zip')
        row['backup_file'] = os.path.relpath(backup, BASE)
        _replace = ctt.annotation_count(task) > 0
        if _replace and not (rb.is_target(task.name) or RL_REPLACE_ANY):   # v4.31.9 (F5): 요청 뒤 사람이 검토해 이름이 바뀐 TASK는 교체하지 않음
            row.update(status='보류', note=f"이름이 {task.name} (uld·AA 아님) — 사람 라벨일 수 있어 교체하지 않음 · 백업 {os.path.basename(backup)} (RL_REPLACE_ANY 또는 10번 셀)")
            L.set(mode='보류'); L.say(row['note'])
            return row
        L.counts(before_zip=backup)
        L.set(backup_file=row['backup_file'], mode='replace' if _replace else 'new')
        L.say(f"백업 {os.path.basename(backup)} · 기존 라벨 박스 {L.rec.get('before_total', 0)}개")
        r = ctt.auto_annotate_frames(task, _RL_FUNC, mpath, imgsz=IMGSZ, device=_rl_dev(), conf=conf, tile=False, quality='original',
                                     frames=None, log=L.say, prefetch=4, replace=_replace, backup=backup,
                                     video=vid, video_offset=off, decoder='pyav')
        res = {'boxes': r['boxes'], 'per_label': r['per_label'], 'frames': r['frames'], 'replaced': r.get('replaced')}
        row.update(status='완료(교체)' if r.get('replaced') is not None else '완료',
                   note=f"재처리 앱 · 박스 {r['boxes']}개 {r['per_label']} · 프레임 {r['frames']}장")
        L.set(uploaded=f"박스 {r['boxes']}개 {r['per_label']}", frames_done=r['frames'], quality_used=r['quality'])
        _lu = label_uploaded(task, replaced=r.get('replaced') is not None, before_zip=backup, say=L.say)   # v4.31.5 이름 규칙 (앱 작업자도 같음)
        if _lu['note']:
            row['note'] += ' · ' + _lu['note']
        if _lu['renamed']:
            row['task_name'] = _lu['name']; L.set(task_name_after=_lu['name'])
        try:
            _ch = change_report(task, backup, 'after_auto', show=False)
            row['note'] += f" · {_ch['text']}"; res['change'] = _ch['text']
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
        except Exception as _e:
            L.say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
    except Exception as e:
        err = e
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [{tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if L is not None:
            try:
                L.finish(row.get('status', '?'), err)
            except Exception:
                pass
        rb.add_history(BASE, tid, kind='shape', model=mlabel, conf=conf, status=row.get('status'), note=row.get('note', ''),
                       backup_file=row.get('backup_file', ''), request=req['id'], who=who_am_i(), **res)
        if str(row.get('status', '')).startswith('완료'):
            _rl_clear_stale(tid, req)
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**')
    return row

def _rl_track_one(tid, mpath, mlabel, conf, req):
    # 5번 셀과 같은 track 생성 (올리지 않음) → 확인 대기 + 폰에서 볼 그림
    folder = task_dir(tid); os.makedirs(folder, exist_ok=True)
    ts = ctt.stamp()
    row = dict(run_time=ctt.now(), action='track', task_id=tid, model=mlabel, server=CVAT_SERVER)
    pend = None
    try:
        task = _rl_cl().tasks.retrieve(tid)
        info = ctt.task_basic(task)
        row.update(task_name=info['task_name'], media=info['media'], frames=info['frames'])
        _uld, _fm = None, None
        if not info['is_video']:
            _uld = uld_task(tid)                           # v4.31.5: ULD 사진 TASK는 clip으로 track
            _why = ('' if uld_is_images(_uld) else 'ULD 기록 없음') or uld_mismatch(task, _uld)
            if _why:
                row.update(status='건너뜀', note=f'이미지 TASK → track 불가 ({_why} · shape를 쓰세요)')
                return row
            _fm = [int(x) for x in _uld['frame_map']]
        backup = ctt.export_annotations(task, f'{folder}/backup_{ts}.zip')
        meta = ctt.read_export(backup)
        row.update(step=meta['step'], existing_labels=meta['existing_boxes'], backup_file=os.path.relpath(backup, BASE))
        if _fm:
            _clip = uld_clip(_uld)
            if _clip is None:
                row.update(status='오류', note='ULD clip을 받지 못함 (CVAT_UpLoader/clips)')
                return row
            video = Path(_clip)
        else:
            video = _rl_video(task, info['task_name'], meta['source'])
        _uvid = False
        if video is None and not _fm:                  # v4.31.8: ULD가 영상으로 올린 TASK → 원본은 ULD clip
            _uc = uld_video(tid)
            if _uc:
                video, _uvid = Path(_uc), True
        if video is None:
            row.update(status='영상 없음', note='백업(영상)이 아직 없음')
            return row
        row['video'] = video.name
        out_dir = f'{folder}/track_{ts}'
        log = f'{folder}/run_{ts}_track.log'
        rc = run([sys.executable, f'{SCRIPTS}/yolo_track_to_cvat.py', '--model', mpath, '--video', video,
                  '--reference', backup, '--imgsz', IMGSZ, '--conf', conf, '--fill-gaps', TRACK_FILL_GAPS,
                  '--track-buffer', TRACK_BUFFER, '--decoder', 'pyav' if (_fm or _uvid) else TRACK_DECODER,
                  '--frame-offset', 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET, *(['--frame-map', ','.join(map(str, _fm))] if _fm else []),
                  '--box-source', TRACK_BOX_SOURCE, '--near', TRACK_NEAR, '--device', _rl_dev(), '--out', out_dir], log=log)
        if rc != 0:
            raise RuntimeError(f'track 생성 실패 (로그 {os.path.relpath(log, BASE)})')
        _mt, _md5 = ctt.file_fingerprint(mpath)
        json.dump({'made': ctt.now(), 'operator': who_am_i(), 'session': SESSION_ID, 'server': CVAT_SERVER,
                   'model': os.path.relpath(mpath, BASE), 'model_time': _mt, 'model_md5': _md5,
                   'video': os.path.relpath(str(video), BASE), 'reference_backup': os.path.relpath(backup, BASE), 'app_version': APP_VERSION,
                   'settings': {'imgsz': IMGSZ, 'conf': conf, 'fill_gaps': TRACK_FILL_GAPS, 'track_buffer': TRACK_BUFFER,
                                'decoder': 'pyav' if (_fm or _uvid) else TRACK_DECODER, 'frame_offset': 0 if (_fm or _uvid) else TRACK_FRAME_OFFSET,   # v4.31.9 (F20)
                                'box_source': TRACK_BOX_SOURCE,
                                'near': TRACK_NEAR, 'app': 'relabel'}},
                  open(f'{out_dir}/track_settings.json', 'w', encoding='utf-8'), ensure_ascii=False, indent=1)
        xml_path = f'{out_dir}/tracks_{video.stem}.xml'
        summ = rb.track_summary(xml_path)
        prev = f'{BASE}/relabel/previews/{tid}_{ts}.jpg'
        try:
            _vg = ctt.video_frame_getter(video)
            pth, _fr = rb.preview_image(xml_path, (lambda k: _vg(_fm[k]) if 0 <= k < len(_fm) else None) if _fm else _vg, prev,
                                        offset=0 if (_fm or _uvid) else TRACK_FRAME_OFFSET)
        except Exception as _e:
            pth = None
            print(f'   [{tid}] (확인 그림을 만들지 못함: {_e})')
        row.update(tracks=summ['tracks'], result_file=os.path.relpath(xml_path, BASE),
                   status='완료 · 확인 대기' if summ['tracks'] else '완료 · track 없음',
                   note=f"재처리 앱 · track {summ['tracks']}개 · 박스 {summ['boxes']}개 {summ['by_label']}")
        if _fm or _uvid:
            uld_clip_done(video)
        if summ['tracks']:
            pend = {'made': ctt.now(), 'model': mlabel, 'model_path': os.path.relpath(mpath, BASE), 'conf': conf,
                    'xml': os.path.relpath(xml_path, BASE), 'video': os.path.relpath(str(video), BASE),
                    'preview': os.path.relpath(pth, BASE) if pth else None, 'request': req['id'], **summ}
    except Exception as e:
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [{tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if pend:
            rb.set_pending_track(BASE, tid, pend)
        rb.add_history(BASE, tid, kind='track', model=mlabel, conf=conf, status=row.get('status'), note=row.get('note', ''),
                       request=req['id'], who=who_am_i(), tracks=row.get('tracks'))
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**', 'relabel/previews/**')
    return row

def _rl_upload_one(tid, req):
    # 6번 셀과 같은 업로드: 직전 백업 → 기존 라벨이 있으면 교체 → 전·후 비교 · 이름 규칙 (v4.31.5: …_uld → …_uld_AA)
    rec = rb.task_record(BASE, tid)
    pend = rec.get('track_pending')
    row = dict(run_time=ctt.now(), action='upload', task_id=tid, server=CVAT_SERVER)
    L, err = None, None
    if not pend:                                   # v4.28.2: 같은 track을 또 올리라는 요청(두 번 누름) → 오류가 아니라 '이미 올림'
        last = [h for h in (rec.get('history') or []) if h.get('kind') == 'upload']
        if last and str(last[-1].get('status', '')).startswith('업로드 완료'):
            row.update(status='이미 올림', note=f"{str(last[-1].get('time', ''))[:16]}에 올림 — 다시 올리지 않음")
            rb.add_history(BASE, tid, kind='upload', status='이미 올림', request=req['id'], who=who_am_i(), note=row['note'])
            print(f"   [업로드 {tid}] 이미 올린 track — 다시 올리지 않음")
            return row
    try:
        if not pend:
            raise RuntimeError('확인 대기 중인 track이 없습니다')
        row.update(model=pend.get('model', ''), tracks=pend.get('tracks', ''), result_file=pend['xml'])
        xml_path = f"{BASE}/{pend['xml']}"
        if ON_KAGGLE and not os.path.exists(xml_path):
            drive_pull(os.path.dirname(pend['xml']), quiet=True)
        ts_ = track_settings_of(pend['xml'])
        L = upload_log(tid, 'track', model_path=f"{BASE}/{pend['model_path']}" if pend.get('model_path') else None,
                       echo=lambda m: print(f'   [업로드 {tid}] {m}'), result_file=pend['xml'], source=pend.get('video', ''),
                       settings={**(ts_.get('settings') or {}), 'request': req['id']}, track_made=pend.get('made', ''))
        task = _rl_cl().tasks.retrieve(tid)
        row['task_name'] = task.name
        L.set(task_name=task.name)
        b = ctt.export_annotations(task, f'{task_dir(tid)}/before_upload_{ctt.stamp()}.zip')
        meta_ = ctt.read_export(b)
        row.update(existing_labels=meta_['existing_boxes'], backup_file=os.path.relpath(b, BASE))
        L.set(backup_file=row['backup_file'], mode='replace' if meta_['existing_boxes'] else 'new')
        L.counts(before_zip=b)
        if meta_['existing_boxes'] > 0 and not (rb.is_target(task.name) or RL_REPLACE_ANY):   # v4.31.9 (F5)
            row.update(status='보류', note=f"이름이 {task.name} (uld·AA 아님) — 사람 라벨일 수 있어 올리지 않음 · track은 확인 대기에 그대로 (6번 셀에서 ALLOW_REPLACE)")
            L.set(mode='보류'); L.say(row['note'])
            return row
        ctt.import_tracks(task, xml_path, replace=meta_['existing_boxes'] > 0, backup=b, log=L.say)
        row.update(status='업로드 완료', note=('기존 라벨 교체(백업 있음) · ' if meta_['existing_boxes'] else '') + '재처리 앱')
        L.set(uploaded=f"track {pend.get('tracks')}개")
        _lu = label_uploaded(task, replaced=meta_['existing_boxes'] > 0, before_zip=b, say=L.say)   # v4.31.5 이름 규칙
        if _lu['note']:
            row['note'] += ' · ' + _lu['note']
        if _lu['renamed']:
            row['task_name'] = _lu['name']; L.set(task_name_after=_lu['name'])
        try:
            _ch = change_report(task, b, 'after_upload', show=False)
            row['note'] += f" · {_ch['text']}"
            L.set(after_file=_ch['after'], change_csv=_ch['csv']); L.counts(after_zip=f"{BASE}/{_ch['after']}")
        except Exception as _e:
            L.say(f'(라벨 변화 표를 만들지 못함: {ctt.api_error(_e)})')
        rb.set_pending_track(BASE, tid, None)
    except Exception as e:
        err = e
        row.update(status='오류', note=ctt.api_error(e)[:300])
        print(f'   [업로드 {tid}] ❌ {row["note"]}')
    finally:
        _rl_reg(**row)
        if L is not None:
            try:
                L.finish(row.get('status', '?'), err)
            except Exception:
                pass
        rb.add_history(BASE, tid, kind='upload', model=row.get('model', ''), status=row.get('status'), note=row.get('note', ''),
                       backup_file=row.get('backup_file', ''), request=req['id'], who=who_am_i())
        _rl_push(f'tasks/{tid}/**', 'task_registry.csv', 'logs/**')
    return row

_RL_ACTION = {'shape': 'auto', 'track': 'track', 'upload': 'upload', 'backup': 'backup'}
_OK = ('완료', '업로드 완료', '이미 올림')

def _rl_ids(req, key='task_ids'):
    return [int(t) for t in (req.get(key) or []) if str(t).strip().isdigit()]

def _rl_short(note):
    # 요청에 남길 TASK 결과 한 줄 (라벨 변화 상세는 TASK 이력에)
    s = str(note or '').replace('재처리 앱 · ', '').replace('재처리 앱', '').strip(' ·')
    return s.split(' · 기존 객체')[0][:140]

def _rl_res(rows):
    # TASK별 결과 → 요청 파일의 results (폰 '요청' 탭이 TASK마다 무엇을 했는지 보여 줌) — v4.29.1
    return [{'task_id': r['task_id'], 'name': r.get('task_name', ''), 'status': r.get('status', ''), 'note': _rl_short(r.get('note'))} for r in rows]

def _rl_merge(req, new):
    # 앞서 남긴 TASK 결과에 새 결과를 덮어 합침 (백업 받은 뒤 track 등)
    old = {str(x.get('task_id')): x for x in (req.get('results') or [])}
    for x in new:
        old[str(x['task_id'])] = {**old.get(str(x['task_id']), {}), **x}
    return list(old.values())

def _rl_do(req):
    # 요청 하나 처리 → (상태, 요약, 추가 칸) · 상태 'pending' = 작업자가 멈춰 남은 TASK를 다시 대기로
    kind, p = req['kind'], dict(req.get('params') or {})
    ids = _rl_ids(req)
    if kind == 'refresh':
        _rl_refresh(force=True); return 'done', '목록을 새로 읽음', {}
    if kind in ('hide', 'unhide'):
        n = rb.set_hidden(BASE, ids, kind == 'hide'); return 'done', f'{n}개 {rb.KINDS[kind]}', {}
    if kind == 'discard':
        for t in ids:
            rb.set_pending_track(BASE, t, None)
            rb.add_history(BASE, t, kind='discard', status='버림', request=req['id'], who=who_am_i())
        return 'done', f'track {len(ids)}개 버림', {'results': [{'task_id': t, 'status': '버림'} for t in ids]}
    if not ids:
        return 'failed', 'TASK가 없습니다', {}
    job_start(30, _RL_ACTION[kind], {'request': req['id'], 'ids': ids, **p}, summary=f"재처리 앱 {rb.KINDS[kind]}")
    rows, extra, left = [], {}, []
    try:
        if kind == 'backup':                       # 요청만 → '받는 중' (받기는 받기 줄이 RL_BACKUP_POLL_MIN마다)
            _rl_lane({'what': '백업 요청', 'request': req['id'], 'tasks': ids, 'since': ctt.now()})
            st = _rl_backup(ids, 'request')
            wait_ = _rl_bk_left(ids, st)
            for t in ids:
                rb.add_history(BASE, t, kind='backup', status='요청함 · 받는 중' if t in wait_ else (st.get(str(t)) or '?'),
                               request=req['id'], who=who_am_i())
            summ = (f"CVAT에 요청함 · 받는 중 {len(wait_)}개" if wait_ else '') + \
                   (' · ' if wait_ and len(wait_) < len(ids) else '') + \
                   ', '.join(f'{k} {v}' for k, v in st.items() if int(k) not in wait_)
            job_end('끝', summ or '바뀐 것 없음')
            res = [{'task_id': t, 'status': '받는 중 (CVAT 준비 중)' if t in wait_ else (st.get(str(t)) or '?')} for t in ids]
            if wait_:
                return 'waiting', summ, {'remaining': wait_, 'waiting_t': _time.time(), 'results': res}
            return 'done', summ or '바뀐 것 없음', {'results': res}
        conf = float(p.get('conf') or AA_CONF)
        if kind in ('shape', 'track'):
            mpath, mlabel = _rl_model(p.get('model'))
        if kind in ('shape', 'track'):             # v4.29.3: TASK를 라벨링 줄들에 나눠 동시에 (다른 요청의 TASK와 함께 RL_GPU_WORKERS개까지)
            job = ctt.current_job()
            def _one(i, t):
                if _RL['stop']:                    # ■: 시작하지 않은 TASK는 다시 대기
                    return None
                ctt._ACT['job'] = job               # 이 줄의 작업 기록을 요청의 작업에 묶음
                try:
                    _rl_lane({'what': rb.KINDS[kind], 'request': req['id'], 'task': t, 'n': i, 'of': len(ids), 'since': ctt.now(), 'device': _rl_dev()})
                    print(f"[{ctt.now()[11:16]}] {rb.KINDS[kind]} {i}/{len(ids)} · TASK {t} · {threading.current_thread().name.replace('rl-gput_', '라벨링 줄 ')}"
                          + (f" (GPU {_rl_dev()})" if _RL_NGPU > 1 else ''))
                    r = (_rl_shape_one if kind == 'shape' else _rl_track_one)(t, mpath, mlabel, conf, req)
                    _RL['dirty'] = True
                    return r
                finally:
                    ctt._ACT['job'] = None
                    _rl_lane(None)
            futs = [_RL_POOLS['task'].submit(_one, i, t) for i, t in enumerate(ids, 1)]
            _fwait(futs)
            for t, f in zip(ids, futs):
                r = f.result()
                if r is None:
                    left.append(t)
                else:
                    rows.append(r)
        else:
            for i, t in enumerate(ids, 1):
                if _RL['stop']:                    # ■: 하던 TASK까지 끝냄 → 남은 TASK는 다시 대기
                    left = ids[i - 1:]
                    break
                _rl_lane({'what': rb.KINDS[kind], 'request': req['id'], 'task': t, 'n': i, 'of': len(ids), 'since': ctt.now()})
                print(f"[{ctt.now()[11:16]}] {rb.KINDS[kind]} {i}/{len(ids)} · TASK {t}")
                rows.append(_rl_upload_one(t, req))
                _RL['dirty'] = True
        if left:
            summ = f"작업자가 멈춰 다시 대기 — {len(rows)}/{len(ids)}개 끝남, 남은 {len(left)}개"
            job_end('멈춤', summ)
            return 'pending', summ, {'task_ids': left, 'done_ids': [r['task_id'] for r in rows], 'results': _rl_merge(req, _rl_res(rows))}
        if kind == 'track':                       # 영상(백업)이 없던 TASK: 백업을 요청만 하고 '받는 중' → 준비되면 GPU 줄이 마저 만듦
            need = [r['task_id'] for r in rows if r.get('status') == '영상 없음']
            if need and p.get('auto_backup', True):
                print(f'   영상이 없는 TASK {len(need)}개 → 백업 요청 (받는 동안 다른 요청을 처리, 준비되면 track을 마저 만듦)')
                _rl_lane({'what': '백업 요청 (track용)', 'request': req['id'], 'tasks': need, 'since': ctt.now()})
                st = _rl_backup(need, 'request')
                wait_ = _rl_bk_left(need, st)
                for t in [x for x in need if x not in wait_]:     # 이미 백업이 있었음 → 바로 다시
                    rows = [r for r in rows if r['task_id'] != t] + [_rl_track_one(t, mpath, mlabel, conf, req)]
                if wait_:
                    extra = {'remaining': wait_, 'waiting_t': _time.time()}
                    for r in rows:
                        if r['task_id'] in wait_:
                            r.update(status='백업 받는 중', note='영상이 없어 TASK 백업을 요청함 → 받으면 track을 만듦')
        ok = sum(1 for r in rows if str(r.get('status', '')).startswith(_OK))
        summ = f"{ok}/{len(ids)}개 완료" + (f" · 백업 받는 중 {len(extra['remaining'])}개" if extra.get('remaining') else '') \
               + ''.join(f" · {r['task_id']} {r.get('status')}" for r in rows if not str(r.get('status', '')).startswith(_OK)
                         and r['task_id'] not in (extra.get('remaining') or []))[:400]
        job_end('끝' if ok or extra else '실패', summ)
        extra['results'] = _rl_res(rows)
        return ('waiting' if extra.get('remaining') else 'done' if ok else 'failed'), summ, extra
    except Exception as e:
        job_end('실패', ctt.api_error(e)[:300])
        raise

def _rl_run(req):
    # 줄(스레드)에서 요청 하나 → 상태 기록
    try:
        st_, summ, extra = _rl_do(req)
    except Exception as e:
        st_, summ, extra = 'failed', ctt.api_error(e)[:400], {}
        traceback.print_exc()
    if st_ == 'pending':
        _rl_status(req, 'pending', result=summ, **extra)
    else:
        _rl_status(req, st_, result=summ, done_at=ctt.now() if st_ != 'waiting' else '', **extra)
    print(f"   → {req['id']} {wb.STATUS_KO.get(st_, st_)} · {summ}")
    return st_

def _rl_bk_poll():
    # '받는 중' 요청의 백업을 받음 (입출력 줄) — 백업 요청은 다 받으면 완료, track 요청은 받은 TASK를 GPU 줄로 넘김
    _alive = _rl_alive()                         # v4.32.0: 다른 살아 있는 세션이 받은 '받는 중'은 손대지 않음 · 꺼진 세션 것은 넘겨받음
    reqs = []
    for r in rb.waiting_requests(BASE):
        if r['id'] in _RL['ready'] or not wb.may_adopt(r, SESSION_ID, _alive):
            continue
        if r.get('taken_session') != SESSION_ID:
            r = _rl_status(r, 'waiting', _adopt=True, taken_session=SESSION_ID, taken_host=_RL['host'], adopted_from=r.get('taken_session') or '')
            if r.get('_skipped'):
                continue
            print(f"   ↪ 받는 중 요청 {r['id']}을 넘겨받음 (앞 세션 {r.get('adopted_from') or '?'}이 꺼짐)")
        reqs.append(r)
    need = sorted({t for r in reqs for t in _rl_ids(r, 'remaining')})
    if not need:
        return
    _rl_lane({'what': '백업 받는 중', 'tasks': need, 'since': ctt.now()})
    try:
        _RL['bk_rows'] = []
        st = _rl_backup(need, 'download')
        # v4.31.10 (R1): 열린 백업 요청이 없어졌다는 것만으로 받음이 아님 — '최신'이고 파일이 있어야 받음 (CPU 마스터와 같은 wb.backup_verdict)
        ok_, wait_, bad_ = (set(map(str, x)) for x in wb.backup_verdict(BASE, _RL.get('bk_rows'), need, server=CVAT_SERVER))
        for r in reqs:
            rem = _rl_ids(r, 'remaining')
            got = [t for t in rem if str(t) in ok_]
            left = [t for t in rem if str(t) in wait_]
            lost = [t for t in rem if str(t) in bad_]
            since = float(r.get('waiting_t') or 0) or _time.time()
            late = left and _time.time() - since > RL_BACKUP_GIVEUP_H * 3600
            if r['kind'] == 'backup':
                for t in got:
                    rb.add_history(BASE, t, kind='backup', status=st.get(str(t)) or '받음', request=r['id'], who=who_am_i())
                for t in lost:
                    rb.add_history(BASE, t, kind='backup', status='실패 (백업이 만들어지지 않음)', request=r['id'], who=who_am_i())
                res = _rl_merge(r, [{'task_id': t, 'status': st.get(str(t)) or '받음'} for t in got]
                                + [{'task_id': t, 'status': '실패 (백업이 만들어지지 않음)'} for t in lost]
                                + ([{'task_id': t, 'status': '포기 (준비 안 됨)'} for t in left] if late else []))
                if late:
                    for t in left:
                        rb.add_history(BASE, t, kind='backup', status='포기 (준비 안 됨)', request=r['id'], who=who_am_i())
                    _rl_status(r, 'failed', remaining=left, done_at=ctt.now(), results=res,
                               result=(str(r.get('result', '')) + f" · {RL_BACKUP_GIVEUP_H}시간 넘게 준비 안 됨 {len(left)}개 → 다시 요청하세요")[:500])
                elif not left:                    # v4.31.11 (R7): 이번 회차가 아니라 쌓인 결과 전체로 — 앞서 실패한 TASK가 있으면 실패
                    _fail = wb.failed_results(res)
                    _rl_status(r, 'failed' if _fail else 'done', remaining=[], done_at=ctt.now(), results=res,
                               result=(str(r.get('result', '')) + (f" · {len(got)}개 받음 · 실패 {len(_fail)}개" if _fail else " · 모두 받음"))[:500])
                elif got or lost:
                    _rl_status(r, 'waiting', remaining=left, waiting_t=since, results=res,
                               result=(str(r.get('result', '')) + f" · {len(got)}개 받음" + (f" · 실패 {len(lost)}개" if lost else ''))[:500])
                elif not r.get('waiting_t'):
                    _rl_status(r, 'waiting', waiting_t=since)
            elif got or late or lost:             # track: 받은 TASK(와 포기·실패한 TASK) → GPU 줄이 마저 함
                _RL['ready'][r['id']] = {'got': got, 'gave_up': (left if late else []) + lost}
            elif not r.get('waiting_t'):
                _rl_status(r, 'waiting', waiting_t=since)
            _RL['dirty'] = True
    finally:
        _rl_lane(None)

def _rl_track_resume(rid):
    # 백업을 받은 TASK의 track을 마저 만듦 (GPU 줄)
    info = _RL['ready'].get(rid) or {}
    req = _rl_req(rid)
    try:
        if not req or req['status'] != 'waiting':
            return 'skip'
        p = dict(req.get('params') or {})
        got, gave = info.get('got') or [], info.get('gave_up') or []
        rem = [t for t in _rl_ids(req, 'remaining') if t not in got and t not in gave]
        mpath = mlabel = None
        if got:                                    # v4.31.12 (R12): 만들 track이 없으면(실패·포기만) 모델을 읽지 않고 바로 마무리
            mpath, mlabel = _rl_model(p.get('model'))
        conf = float(p.get('conf') or AA_CONF)
        bad, rows = [], []
        for i, t in enumerate(got, 1):
            _rl_lane({'what': 'track 만들기 (백업 받은 뒤)', 'request': rid, 'task': t, 'n': i, 'of': len(got), 'since': ctt.now()})
            print(f"[{ctt.now()[11:16]}] track 만들기 (백업 받은 뒤) {i}/{len(got)} · TASK {t}")
            r = _rl_track_one(t, mpath, mlabel, conf, req)
            rows.append(r)
            if not str(r.get('status', '')).startswith('완료'):
                bad.append(f"{t} {r.get('status')}")
            _RL['dirty'] = True
        note = f" · 백업 뒤 {len(got) - len(bad)}개 track 완료" + ''.join(f" · {b}" for b in bad) \
               + (f" · 백업 포기 {len(gave)}개" if gave else '')
        res = _rl_merge(req, _rl_res(rows) + [{'task_id': t, 'status': '백업 포기 (준비 안 됨)', 'note': ''} for t in gave])
        if rem:
            _rl_status(req, 'waiting', remaining=rem, results=res, result=(str(req.get('result', '')) + note)[:500])
        else:
            _rl_status(req, 'done' if len(got) > len(bad) or not gave and not bad else 'failed', remaining=[], done_at=ctt.now(), results=res,
                       result=(str(req.get('result', '')) + note)[:500])
        return 'done'
    except Exception as e:                         # v4.31.12 (R12): 마무리 중 오류(모델 없음 등)도 요청에 실패로 남김 — '받는 중'으로 계속 남지 않게
        traceback.print_exc()
        try:
            _cur = _rl_req(rid)
            if _cur and _cur['status'] == 'waiting':
                _ids = [t for t in _rl_ids(_cur, 'remaining')]
                _res = _rl_merge(_cur, [{'task_id': t, 'status': '실패 (마무리 오류)', 'note': _rl_short(ctt.api_error(e))} for t in _ids])
                _rl_status(_cur, 'failed', remaining=[], done_at=ctt.now(), results=_res,
                           result=(str(_cur.get('result', '')) + f" · 백업 뒤 마무리 실패: {ctt.api_error(e)}")[:500])
        except Exception:
            traceback.print_exc()
        return 'failed'
    finally:
        _RL['ready'].pop(rid, None)

# ---------- 차례 정하기 ----------
def _rl_done(fut, lane, ids):
    with _RL_LOCK:
        for t in ids:
            _RL['tasks'].pop(t, None)
        _RL['dirty'] = True
    _RL_EV.set()

def _rl_submit(lane, fn, arg, ids, label, who=None):
    with _RL_LOCK:
        for t in ids:
            _RL['tasks'][t] = label
    def _go():
        try:
            return fn(arg)
        finally:
            _rl_lane(None)
    f = _RL_POOLS[lane].submit(_go)
    _RL_POOLS['futs'][f] = (lane, label)
    _RL_POOLS.setdefault('who', {})[f] = who        # v4.31.6: 한 사람이 무거운 일을 W_USER_MAX건 넘게 차지하지 않게
    f.add_done_callback(lambda f_, l=lane, i=list(ids): _rl_done(f_, l, i))
    return f

def _rl_user_running(u):
    return sum(1 for f, w in list(_RL_POOLS.get('who', {}).items()) if w == u and not f.done())

def _rl_running(lane):
    return [f for f, (l, _) in list(_RL_POOLS['futs'].items()) if l == lane and not f.done()]

def _rl_claim(r):
    # v4.32.0: 요청을 받고(taken) 잠시 뒤 다시 읽어 확인 — 다른 작업자가 거의 동시에 받았으면 물러남 → 받은 요청 또는 None
    r2 = _rl_status(r, 'taken', taken_by=who_am_i(), taken_session=SESSION_ID, taken_at=ctt.now(), taken_host=_RL['host'], host=_RL['host'])
    if r2.get('_skipped'):
        return None
    if W_CLAIM_CONFIRM_SEC:
        _time.sleep(W_CLAIM_CONFIRM_SEC)
    cur = _req_remote(r['id'])
    if not cur or cur.get('status') != 'taken' or cur.get('taken_session') != SESSION_ID:
        print(f"ℹ️ 요청 {r['id']}: 다른 작업자가 거의 동시에 받음 → 물러남")
        return None
    return r2

def _rl_dispatch(poll=True):
    # 대기 요청을 차례대로 보며 맡길 수 있는 것을 맡김 → 맡긴 수
    n = 0
    _own = _own_state()                          # v4.32.0: 숨기기·다시 보이기·새로 고침(tracked·상태를 고침)은 취합자만 (배정과 관계없이)
    reqs = [r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS
            and (_own if r['kind'] in _RL_LIGHT else _w_mine(r['id'], _RL['host']))]   # v4.31.4: 배정(requests/assign) 따르기
    reqs = wb.fair_order(reqs, _RL_LIGHT, _w_served())      # v4.31.6: 공정 차례 (가벼운 일 → 사람마다 돌아가며 → 오래 기다린 것)
    busy, blocked = set(_RL['tasks']), set()
    gpu_free = max(1, int(RL_GPU_WORKERS)) - len(_rl_running('gpu')) - int(_W['lc_gpu'] or 0)   # LC가 GPU 줄을 맡아 둠 (학습은 모두)
    io_free = max(1, int(RL_IO_WORKERS)) - len(_rl_running('io'))
    users = {wb._user_of(r) for r in reqs}
    for rid in list(_RL['ready']):                # 백업을 받은 track 요청 (먼저 보낸 요청이므로 먼저)
        rq = _rl_req(rid)
        ids = set(_RL['ready'][rid].get('got') or [])
        if rq is None or gpu_free <= 0 or ids & busy:
            if rq is None:
                _RL['ready'].pop(rid, None)
            blocked |= ids
            continue
        _rl_submit('gpu', _rl_track_resume, rid, sorted(ids), f'track 마저 {rid}'); n += 1
        gpu_free -= 1; busy |= ids
    for r in reqs:
        k, ids = r['kind'], set(_rl_ids(r))
        if k in _RL_LIGHT or (k == 'discard' and not ids & (busy | blocked)):
            print(f"\n📨 [{ctt.now()[11:16]}] {rb.KINDS.get(k, k)} · {r.get('by', '')} · TASK {len(ids)}개 · {r['id']}")
            r = _rl_claim(r)
            if r is None:
                continue
            _rl_run(r); n += 1; _RL['dirty'] = True
            continue
        lane = 'gpu' if k in _RL_GPU else 'io' if k in _RL_IO else None
        if lane is None or ids & (busy | blocked) or (lane == 'gpu' and gpu_free <= 0) or (lane == 'io' and io_free <= 0):
            blocked |= ids                        # 같은 TASK의 뒤 요청은 이 요청이 끝난 뒤에
            continue
        u = wb._user_of(r)
        if W_USER_MAX and len(users) > 1 and _rl_user_running(u) >= int(W_USER_MAX):
            blocked |= ids                        # v4.31.6: 이 사람은 이미 W_USER_MAX건 — 다른 사람 요청 먼저
            continue
        print(f"\n📨 [{ctt.now()[11:16]}] {rb.KINDS.get(k, k)} · {r.get('by', '')} · TASK {len(ids)}개 · {r['id']} → {'GPU 줄' if lane == 'gpu' else '입출력 줄'}")
        r = _rl_claim(r)
        if r is None:
            continue
        _rl_submit(lane, _rl_run, r, sorted(ids), f"{rb.KINDS.get(k, k)} {r['id']}", who=u); n += 1
        _w_served_add(u)
        busy |= ids
        if lane == 'gpu':
            gpu_free -= 1
        else:
            io_free -= 1
    if poll and io_free > 0 and _time.time() - _RL['bk_t'] >= RL_BACKUP_POLL_MIN * 60 \
            and not any(lbl == '백업 받기' for f, (l, lbl) in list(_RL_POOLS['futs'].items()) if not f.done()):
        if any(_rl_ids(r, 'remaining') for r in rb.waiting_requests(BASE) if r['id'] not in _RL['ready']):
            _RL['bk_t'] = _time.time()
            f = _RL_POOLS['io'].submit(_rl_bk_poll)
            _RL_POOLS['futs'][f] = ('io', '백업 받기')
            f.add_done_callback(lambda f_: _RL_EV.set())
            n += 1
    return n

def _rl_wait_all():
    _fwait([f for f in list(_RL_POOLS['futs']) if not f.done()])
    _rl_tidy()

def _rl_tidy():
    for f in [f for f in list(_RL_POOLS['futs']) if f.done()]:
        _RL_POOLS['futs'].pop(f, None)
        _RL_POOLS.get('who', {}).pop(f, None)

_rl_left = []
for _r in [r for r in rb.list_requests(BASE) if r['status'] == 'taken']:   # 앞 작업자가 멈추며 남긴 '진행 중' → 다시 대기 (처음부터 다시 함, 올리기 전 백업이 있음)
    if (_r.get('taken_session') == SESSION_ID or (_RL_PREV.get('session') and _r.get('taken_session') == _RL_PREV.get('session'))
            or wb.age_hours(_r.get('taken_at')) >= RL_STALE_TAKEN_H):          # v4.31.4: 다른 세션(CPU 마스터 등)이 진행 중인 것은 그대로
        _rl_status(_r, 'pending', _adopt=_r.get('taken_session') != SESSION_ID, result='작업자가 멈춰 다시 대기')
        print(f"↺ 멈춘 요청을 다시 대기로: {_r['id']} ({rb.KINDS.get(_r['kind'], _r['kind'])})")
    else:
        _rl_left.append(_r)
if _rl_left:
    print(f"ℹ️ 다른 세션이 진행 중인 요청 {len(_rl_left)}건은 그대로 둡니다: " + ', '.join(f"{r['id']}({r.get('taken_by', '?')})" for r in _rl_left)
          + f" — 끊긴 것이면 {RL_STALE_TAKEN_H}시간 뒤 다시 대기로")
for _t in rb.stale_pending(BASE):                  # v4.29: shape로 라벨이 바뀐 뒤에도 남은 track 확인 대기 → 정리
    _rl_clear_stale(_t)


# =============== LC (예전 31번 셀) ===============
import json, threading, time as _time, traceback, re as _re
from IPython import get_ipython
assert wb is not None and hasattr(wb, 'lc_worker_beat'), 'scripts/를 v4.26.0 이상으로 바꾸세요'
_lc_cells_p = f'{SCRIPTS}/notebook_cells.json'
assert os.path.exists(_lc_cells_p), 'scripts/notebook_cells.json이 없습니다 — v4.26.0 scripts zip을 모두 올리세요'
_LC_CELLS = json.load(open(_lc_cells_p, encoding='utf-8'))
if _LC_CELLS.get('version') not in SCRIPTS_OK:
    print(f"⚠️ notebook_cells.json 버전 {_LC_CELLS.get('version')} ≠ 노트북 {APP_VERSION} — scripts를 노트북과 같은 버전으로 맞추세요")
_LC = {'busy': None, 'stop': False, 'started': ctt.now(), 'host': 'kaggle_gpu' if W_ROLE == 'gpu' else 'Kaggle' if ON_KAGGLE else 'Colab', 'dl_t': 0, 'state_t': 0, 'dirty': False}
if ON_KAGGLE:
    drive_pull('requests/*.json', f'web/worker{W_SLOT}.json', quiet=True)
_LC_PREV = wb.lc_worker_acquire(BASE, who_am_i(), SESSION_ID, takeover=LC_TAKEOVER, slot=W_SLOT) or {}   # 앞 작업자 (멈춘 요청을 되돌릴 때 씀)

def _lc_src(cell, set_=None):
    # 셀 코드 + 설정 바꾸기 (그 셀의 '설정 이름 = …' 첫 줄을 바꿈)
    src = _LC_CELLS['cells'][str(cell)]
    for k, v in {**(set_ or {}), **(LC_SET.get(str(cell)) or {})}.items():
        src, n = _re.subn(rf'(?m)^({_re.escape(k)}\s*=\s*)[^#\n]*?(\s*(#.*)?)$', lambda m: m.group(1) + repr(v) + ('  ' + m.group(3) if m.group(3) else ''), src, count=1)
        if not n:
            print(f'⚠️ {cell}번 셀에 {k} 설정이 없습니다 (무시)')
    return src

def _lc_run(cell, set_=None):
    # 셀 하나를 실행 (Kaggle은 셀마다 Drive와 주고받음) → 오류 (없으면 None) · 셀 중지(■)는 작업자도 멈춤
    print(f"\n━━━━ {cell}번 셀 실행 ({ctt.now()[11:16]}) ━━━━")
    r = get_ipython().run_cell(_lc_src(cell, set_), store_history=False)
    err = r.error_in_exec or r.error_before_exec
    if isinstance(err, KeyboardInterrupt):
        raise err
    return err

def _lc_kg_begin():
    # Kaggle: 요청 받기 + 지금 상태 기억 (작업자가 직접 바꾼 요청 파일만 Drive에 올리고, 이름이 바뀐 예전 파일은 Drive에서 지움)
    if ON_KAGGLE:
        drive_pull('requests/*.json', quiet=True)
        _KG['snap'], _KG['snap_t'] = _kg_snapshot(), _time.time()

def _lc_kg_end():
    if ON_KAGGLE and _KG.get('snap') is not None:
        drive_sync_now(echo=False)

def _lc_beat(busy=None, stopping=False):
    if busy is not None:
        _LC['busy'] = busy
    wb.lc_worker_beat(BASE, who_am_i(), SESSION_ID, _LC['host'], busy=dict(_LC['busy'] or {}, loop_at=_LC.get('loop_at')),   # loop_at: 요청 확인 루프가 마지막으로 돈 시각 (v4.29.4)
                      started=_LC['started'], stopping=stopping, slot=W_SLOT)
    if ON_KAGGLE:
        drive_push(f'web/worker{W_SLOT}.json', quiet=True)

def _lc_heart(st=_LC, beat=_lc_beat, sleep=_time.sleep):
    while not st['stop']:
        try:
            beat()
        except Exception:
            pass
        for _ in range(60):
            if st['stop']:
                return
            sleep(1)

def _lc_fail_left(ids, why):
    # 셀이 요청을 끝내지 못하고 멈췄을 때 (오류·대상 없음): '진행 중'으로 남은 요청을 '실패'로 + 이유
    for r in wb.list_requests(BASE):
        if r['id'] in ids and r['status'] == 'taken':
            wb.set_status(BASE, r, 'failed', result=why[:400], done_at=ctt.now(), by=who_am_i(), summary=why[:200])
            print(f"   → 요청 {r['id']} 실패로: {why[:120]}")

_LC_CELL = {'backup': ['13'], 'export': ['14'], 'track': ['5'], 'aa': ['9', '10'], 'cleanup': ['29'], 'compare': ['32'], 'train': ['15']}

def _lc_claim(ids):
    # v4.32.0: LC 요청을 받고(taken) Drive에 바로 올린 뒤, 잠시 뒤 다시 읽어 이 세션 것만 남김
    got = []
    for r in [r for r in wb.list_requests(BASE) if r['id'] in ids]:
        if r['status'] == 'taken' and r.get('taken_session') == SESSION_ID:
            got.append(r); continue
        if r['status'] != 'pending':
            continue
        new = wb.set_status(BASE, r, 'taken', taken_by=who_am_i(), taken_session=SESSION_ID, taken_at=ctt.now(),
                            taken_host=_LC['host'], by=who_am_i(), host=_LC['host'])
        if ON_KAGGLE:
            _kg_put(f"requests/{os.path.basename(new['_path'])}", new['_path'])
            _kg_rc(['deletefile', _kg_remote(f"requests/{os.path.basename(r['_path'])}")], quiet=True)
        got.append(new)
    if not got:
        return []
    if W_CLAIM_CONFIRM_SEC:
        _time.sleep(W_CLAIM_CONFIRM_SEC)
    out = []
    for r in got:
        name = os.path.basename(r['_path'])
        if ON_KAGGLE:
            tmp = f'{LOCAL_TMP}/_lcreq_{SESSION_ID}.json'
            ok_ = _kg_fetch(f'requests/{name}', tmp) == 'ok'
            cur = json.load(open(tmp, encoding='utf-8')) if ok_ else None
        else:
            cur = next((x for x in wb.list_requests(BASE) if x['id'] == r['id']), None)
        if cur and cur.get('status') == 'taken' and cur.get('taken_session') == SESSION_ID:
            out.append(r['id'])
        else:
            print(f"ℹ️ LC 요청 {r['id']}: 다른 작업자가 거의 동시에 받음 → 물러남")
    return out

def _lc_do(kind):
    ids = [r['id'] for r in wb.list_requests(BASE) if r['kind'] == kind and r['status'] in ('pending', 'taken')
           and (r.get('taken_session') == SESSION_ID if r['status'] == 'taken' else _w_mine(r['id'], _LC['host']))]   # 셀이 받을 것 (v4.31.7: 다른 세션이 진행 중인 것은 빼고)
    ids = _lc_claim(ids)                         # v4.32.0: 셀이 받기 전에 먼저 받고 확인 — 다른 작업자가 받은 것은 빼고
    if not ids:
        return
    _lc_beat({'what': wb.KINDS[kind][0], 'cell': _LC_CELL[kind][0], 'requests': len(ids), 'since': ctt.now()})
    err = None
    for c in _LC_CELL[kind]:
        err = _lc_run(c, {'FROM_REQUESTS': True} if c != '10' else {'RUN_AA': True})
        if err is not None:
            break
    if err is not None:
        why = f"{type(err).__name__}: {err}" if not isinstance(err, SystemExit) else str(err)
    elif kind == 'aa' and not globals().get('AA_READY'):
        why = '자동 라벨을 올릴 TASK가 없음 (9번 셀 점검에서 모두 빠짐 — 기존 라벨이 있거나 영상·프레임 문제)'
    else:
        why = '셀은 끝났지만 요청을 마치지 않음 (출력 확인)'
    _lc_kg_begin()
    _lc_fail_left(ids, why)
    _lc_kg_end()
    _LC['dirty'] = True

def _lc_waiting():
    if W_ROLE == 'gpu':                            # v4.31.7: GPU 워커는 받기·상태 만들기를 하지 않음 (CPU 마스터·30번 셀 몫)
        return
    # '받는 중' 백업·export: CVAT가 준비한 것을 받음 (13·14번 셀 받기 방식) → 다 받으면 요청이 완료로
    if _time.time() - _LC['dl_t'] < LC_DOWNLOAD_MIN * 60:
        return
    _LC['dl_t'] = _time.time()
    w = {r['kind'] for r in wb.list_requests(BASE) if r['status'] == 'waiting'}
    p = {x['kind'] for x in (wb.pending_open(BASE) if hasattr(wb, 'pending_open') else [])}
    if 'backup' in w | p:
        _lc_beat({'what': '백업 받기 (준비된 것)', 'cell': '13', 'since': ctt.now()})
        _lc_run('13', {'FROM_REQUESTS': False, 'BK_MODE': 'download', 'BK_PREVIEW': False, 'BK_QUERY': '', 'BK_TASK_IDS': None})
        _LC['dirty'] = True
    if 'export' in w | p:
        _lc_beat({'what': 'export 받기 (준비된 것)', 'cell': '14', 'since': ctt.now()})
        _lc_run('14', {'FROM_REQUESTS': False, 'EX_MODE': 'download', 'EX_PREVIEW': False, 'EX_QUERY': '', 'EX_IDS': []})
        _LC['dirty'] = True

def _lc_state(force=False):
    if W_ROLE == 'gpu':
        return
    if not (force or _LC['dirty'] or _time.time() - _LC['state_t'] >= LC_STATE_MIN * 60):
        return
    if not _own_state():                           # v4.32.0 (F3·F14): web/state.json은 취합자만 — 아니면 표시만
        if force or _LC['dirty']:
            _dirty_mark('LC 작업 뒤')
        _LC['state_t'], _LC['dirty'] = _time.time(), False
        return
    _lc_beat({'what': 'LC앱 상태 새로 만들기', 'cell': '28', 'since': ctt.now()})
    _lc_run('28', {'LOGIN': LC_LOGIN})
    _LC['state_t'], _LC['dirty'] = _time.time(), False

_lc_kg_begin()
def _lc_age_h(t):
    return wb.age_hours(t)                     # v4.31.4: 시간대 없는 KST 시각('2026-11-07 15:00:00')도 바르게
# '진행 중'으로 남은 요청: 이 세션·앞 LC 작업자가 받다 멈춘 것, 또는 LC_STALE_TAKEN_H시간이 지난 것만 다시 대기로
# (다른 사람이 셀을 직접 실행해 진행 중인 요청은 그대로 둠 — 두 번 실행하지 않게)
_left = []
for _r in [r for r in wb.list_requests(BASE) if r['status'] == 'taken' and r['kind'] in LC_KINDS]:
    if (_r.get('taken_session') == SESSION_ID or (_LC_PREV.get('session') and _r.get('taken_session') == _LC_PREV.get('session'))
            or _lc_age_h(_r.get('taken_at')) >= LC_STALE_TAKEN_H):
        wb.set_status(BASE, _r, 'pending', result='작업자가 멈춰 다시 대기', by=who_am_i())
        print(f"↺ 멈춘 요청을 다시 대기로: {_r['id']} ({wb.KINDS.get(_r['kind'], (_r['kind'],))[0]})")
    else:
        _left.append(_r)
if _left:
    print(f"ℹ️ 다른 세션이 진행 중인 요청 {len(_left)}건은 그대로 둡니다: " + ', '.join(f"{r['id']}({r.get('taken_by', '?')})" for r in _left)
          + f" — 끊긴 것이면 {LC_STALE_TAKEN_H}시간 뒤 다시 대기로 (또는 LC앱에서 확인)")
_lc_kg_end()


# ---------- 돌기 (v4.31.6): 재처리는 뒤 스레드가 요청을 나눠 주고, LC 셀은 이 스레드에서 차례로 ----------
def _w_now_utc():
    import datetime as _d
    return _d.datetime.now(_d.timezone.utc).isoformat(timespec='seconds')

def _w_aa_loop():
    # 재처리 요청 확인 (예전 30번 셀의 돌기) — LC 셀이 오래 걸려도 폰 요청은 바로 받음
    n = 0
    try:
        while not _W['stop'] and (W_MAX_LOOPS is None or n < W_MAX_LOOPS):
            n += 1
            _RL['loop_at'] = _w_now_utc()
            _RL_EV.clear()
            try:
                if ON_KAGGLE:
                    with _RL_LOCK:
                        drive_pull('relabel/requests/*.json', 'requests/assign/*.json', quiet=True)
                got = _rl_dispatch()
                if W_MAX_LOOPS is not None:            # 시험: 맡긴 일이 끝나기를 기다리고, 그 사이 풀린 요청도 이어서
                    while got:
                        _rl_wait_all(); _rl_state(force=True)
                        got = _rl_dispatch(poll=False)
                _rl_state()
                running = [f for f in list(_RL_POOLS['futs']) if not f.done()]
                _rl_tidy()
                if not running and not got:
                    _rl_refresh()
                _RL['errs'] = 0
            except Exception as _e:                    # 요청 확인 중 오류 한 번으로 멈추지 않게 — 알리고 다음 차례에 다시
                _RL['errs'] = _RL.get('errs', 0) + 1
                print(f"⚠ [{ctt.now()[11:16]}] 재처리 요청 확인 중 오류 — 계속 ({_RL['errs']}번 연속): {type(_e).__name__}: {_e}"[:400])
                if _RL['errs'] >= 10:                    # 계속 실패하면 멈춤 (Drive 연결이 끊겼을 때 등)
                    _W['aa_err'] = _e
                    return
            if not _W['stop'] and (W_MAX_LOOPS is None or n < W_MAX_LOOPS):
                _RL_EV.wait(W_POLL_SEC)
    finally:
        _W['aa_done'] = True

def _w_wait_gpu(kind=''):
    # LC의 GPU 일: 한 줄을 맡아 두고(재처리는 새 GPU 일을 덜 받음) 빈 줄이 날 때까지 기다림 · 학습은 모든 줄 (v4.31.7)
    n = max(1, int(RL_GPU_WORKERS))
    _W['lc_gpu'] = n if kind == 'train' else 1
    t0, said = _time.time(), False
    while len(_rl_running('gpu')) > n - _W['lc_gpu'] and not _W['stop']:
        if not said:
            print(f"   (GPU 줄 {len(_rl_running('gpu'))}개가 재처리 라벨링 중 — 하나가 끝나면 시작)")
            said = True
        _lc_beat({'what': 'GPU 줄이 비기를 기다림', 'since': ctt.now()})
        _time.sleep(3)
    if said:
        print(f"   (GPU 줄 비움 — {_time.time() - t0:.0f}초 기다림)")

def _w_status():
    if _time.time() - _W.get('status_t', 0) < W_STATUS_MIN * 60:
        return
    _W['status_t'] = _time.time()
    _a = [r for r in rb.list_requests(BASE) if r['status'] in ('pending', 'taken', 'waiting')]
    _l = [r for r in wb.list_requests(BASE) if r['status'] in ('pending', 'taken', 'waiting')]
    _run = [lbl for f, (l, lbl) in list(_RL_POOLS['futs'].items()) if not f.done()]
    print(f"[{ctt.now()[11:16]}] 대기 중 · 열린 요청 재처리 {len(_a)}건 · LC {len(_l)}건 (학습 대기 {sum(r['kind'] == 'train' and r['status'] == 'pending' for r in _l)}"
          f" · 받는 중 {sum(r['status'] == 'waiting' for r in _a + _l)})" + (f" · 하는 일 {len(_run)}건" if _run else '')
          + f" — 요청 파일 {len(glob.glob(f'{BASE}/relabel/requests/REQ_*.json'))}개 (폰에서 보낸 요청이 안 잡히면 Drive 연결·폴더를 확인)")

_t_end, _wn = _time.time() + W_HOURS * 3600, 0
_aa_th = threading.Thread(target=_w_aa_loop, name='aa-loop', daemon=True)
try:
    threading.Thread(target=_rl_heart, daemon=True).start()   # 멈출 때(finally) 함께 멈춤
    threading.Thread(target=_lc_heart, daemon=True).start()
    _lc_beat({'what': '대기 중'})
    print(f"▶ 작업자 켬 · {_RL['host']} · {who_am_i()} · 최대 {W_HOURS}시간 · 요청 확인 {W_POLL_SEC}초마다 · CVAT {CVAT_SERVER} ({ctt.scope_text(CVAT_SCOPE)})\n"
          f"   재처리(AA앱(가지 재처리)): 라벨링 줄 {max(1, int(RL_GPU_WORKERS))}" + (f" (GPU {_RL_NGPU}개에 나눔)" if _RL_NGPU > 1 else "")
          + f" + 입출력 줄 {max(1, int(RL_IO_WORKERS))}\n"
          f"   LC(LC앱): {', '.join(wb.KINDS[k][0] for k in LC_KINDS)}" + (f" · GPU 워커: 할 일이 없으면 {W_EXIT_IDLE_MIN}분 뒤 끝남" if W_ROLE == 'gpu' else '') + "\n"
          f"   공정 차례: 가벼운 일 → 사람마다 돌아가며 한 건씩(오래 기다린 것 먼저) · 다른 사람이 기다릴 때 한 사람 동시 {W_USER_MAX or '제한 없음'}건")
    _rl_refresh(force=True)
    _aa_th.start()
    while _time.time() < _t_end and (W_MAX_LOOPS is None or _wn < W_MAX_LOOPS):
        _wn += 1
        if _W['aa_err'] is not None:
            raise RuntimeError(f"재처리 요청 확인이 10번 연속 실패해 멈춥니다: {_W['aa_err']}")
        _LC['loop_at'] = _w_now_utc()
        if ON_KAGGLE:
            drive_pull('requests/*.json', 'requests/assign/*.json', 'backups/pending_backups.csv', 'exports/pending_exports.csv',
                       *(['jobs/STOP_GPU.json'] if W_ROLE == 'gpu' else []), quiet=True)
        if W_ROLE == 'gpu' and os.path.exists(f'{BASE}/jobs/STOP_GPU.json'):
            print(f"🛑 [{ctt.now()[11:16]}] CPU 마스터의 정지 신호(jobs/STOP_GPU.json) → 하던 일만 마치고 끝냄")
            break
        _pend = [dict(r, _lc=True) for r in wb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in LC_KINDS
                 and _w_mine(r['id'], _LC['host'])]   # 다른 곳(kaggle_cpu 등)으로 배정된 요청은 건너뜀
        if _pend:
            _aa_pend = [r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS]
            _first = next(r for r in wb.fair_order(_pend + _aa_pend, list(_RL_LIGHT) + ['cleanup'], _w_served()) if r.get('_lc'))
            _k = _first['kind']
            _w_served_add(wb._user_of(_first))                        # 그 사람의 차례인 LC 요청의 종류 → 그 셀이 같은 종류의 요청을 모두 처리
            print(f"\n📨 [{ctt.now()[11:16]}] LC {wb.KINDS[_k][0]} 요청 {sum(1 for r in _pend if r['kind'] == _k)}건 (차례: {_first.get('by', '?')} · {_first['id']}) → {', '.join(_LC_CELL[_k])}번 셀")
            try:
                if _k in LC_GPU_KINDS:
                    _w_wait_gpu(_k)
                _lc_do(_k)
            except KeyboardInterrupt:
                raise
            except Exception as _e:
                traceback.print_exc()
                _lc_kg_begin()
                _lc_fail_left([r['id'] for r in _pend if r['kind'] == _k], f'작업자 오류: {_e}')
                _lc_kg_end()
            finally:
                _W['lc_gpu'] = 0
                _RL_EV.set()                           # 재처리 쪽이 바로 다음 GPU 일을 보게
            _lc_state(force=True)
            _lc_beat({'what': '대기 중'})
            _W['idle_since'] = None
            continue
        if W_EXIT_IDLE_MIN:                            # v4.31.7: 할 일이 없으면 끝냄 (GPU 워커 → GPU 반환)
            _op = {str(x.get('task_id')) for x in wb.pending_rows(BASE, 'backup') if x.get('status') in _RL_OPEN_BK}
            _busy = [f for f in list(_RL_POOLS['futs']) if not f.done()] or _RL['ready'] or [
                r for r in rb.list_requests(BASE) if r['status'] == 'pending' and r['kind'] in rb.KINDS and r['kind'] not in _RL_LIGHT
                and _w_mine(r['id'], _RL['host'])] or [
                r for r in rb.list_requests(BASE) if r['status'] == 'waiting' and r.get('taken_session') == SESSION_ID    # v4.31.9 (F10): 백업이 준비된 '받는 중' track은 마저 처리
                and any(str(t) not in _op for t in (r.get('remaining') or []))]
            if _busy:
                _W['idle_since'] = None
            elif _W['idle_since'] is None:
                _W['idle_since'] = _time.time()
            elif _time.time() - _W['idle_since'] >= float(W_EXIT_IDLE_MIN) * 60:
                print(f"✅ [{ctt.now()[11:16]}] 할 일이 {W_EXIT_IDLE_MIN}분 동안 없어 끝냅니다" + (' (GPU 반환)' if W_ROLE == 'gpu' else ''))
                break
        _owner_marks()                             # v4.32.0: 취합자면 다른 작업자의 표시를 보고 상태를 새로
        if ON_KAGGLE and rb.flush_remote():        # v4.32.0: 못 올린 TASK 기록 고치기를 다시
            print(f"⚠ TASK 기록 {len(rb._REMOTE['queue'])}건을 아직 Drive에 올리지 못함 — 다음 차례에 다시")
        _lc_waiting()
        _lc_state()
        _lc_beat({'what': '대기 중'})
        _w_status()
        if W_MAX_LOOPS is None or _wn < W_MAX_LOOPS:
            _time.sleep(W_POLL_SEC)
    if W_MAX_LOOPS is not None:                        # 시험: 재처리 쪽도 다 돌기를 기다림
        while not _W['aa_done']:
            _time.sleep(0.2)
finally:
    _W['stop'] = True
    _RL_EV.set()
    _LC['stop'] = True
    try:
        _lc_beat({'what': '꺼짐'}, stopping=True)
    except Exception:
        pass
    try:
        _own_state(stopping=True)                  # v4.32.0: 취합자였으면 내려놓음 → 다음 순위가 곧 이어받음
    except Exception:
        pass
    try:
        if _aa_th.is_alive():
            _aa_th.join(timeout=120)               # 지금 나눠 주던 차례만 마치고 끝남
    except BaseException:
        pass
    _RL['stop'] = True
    _RL['loop_at'], _RL['closing'] = None, True
    _left = [f for f in list(_RL_POOLS['futs']) if not f.done()]
    try:
        _rl_beat()                                 # 폰에 '멈추는 중' 바로
    except Exception:
        pass
    try:
        if _left:
            print(f"⏸ 하던 일 {len(_left)}건 — 지금 TASK까지 끝내고 멈춥니다 (남은 TASK는 요청에 '대기'로 남김) …")
            _fwait(_left)
        try:
            _rl_state(force=True)
        except KeyboardInterrupt:                  # v4.31.4: 마지막 상태 파일 쓰기 중 ■ → 조용히 건너뜀 (기록은 이미 Drive에)
            print('ℹ️ 마지막 화면 상태 파일은 건너뜀 — 다음에 켤 때 다시 씀')
        except Exception as _e:
            print(f'⚠ 마지막 상태 파일 쓰기 실패 (무시): {_e}')
    except BaseException:
        print("⚠ 기다리지 않고 멈춤 — 하던 일은 뒤에서 마저 끝나고, 30번 셀을 다시 켜면 그것이 끝난 뒤 시작합니다")
        raise
    finally:
        if W_ROLE == 'gpu':                       # v4.31.7: CPU 마스터가 상태를 새로 만들도록 끝 표시
            try:
                os.makedirs(f'{BASE}/jobs', exist_ok=True)
                json.dump({'session': SESSION_ID, 'time': ctt.now(), 'who': who_am_i()}, open(f'{BASE}/jobs/DONE_{SESSION_ID}.json', 'w'))
                drive_push(f'jobs/DONE_{SESSION_ID}.json', quiet=True)
            except Exception as _e:
                print(f'(끝 표시 실패: {_e})')
        if not [f for f in list(_RL_POOLS['futs']) if not f.done()]:
            _RL['heart_stop'] = True
            try:
                _rl_beat('꺼짐', stopping=True)
            except Exception:
                pass
            print(f"■ 작업자 끔 ({ctt.now()[11:16]})")
